# AI Workforce Statements: Historical Firm Crosswalk

## Part 0 Notebook Setup and Research Scope

**Purpose.** Prepare a transparent historical identifier crosswalk for firms represented
in the S&P 500 from **January 1, 2018 through December 31, 2025**.
The intended identifier relationships are:

**CRSP PERMNO / PERMCO / CCM / Compustat GVKEY / CIK / Revelio RCID**

This notebook currently contains only setup and initial input diagnostics. It does not
construct matches, select preferred records, change membership dates, or export data.
Each subsequent Part requires researcher review and approval.

### Sources and units of observation

| Source | Current unit of observation | Relevant identifiers |
|---|---|---|
| CRSP historical S&P 500 extract | Daily security observations with membership dates | PERMNO, PERMCO, CUSIP, ticker |
| CCM link history (`ccmxpf_lnkhist`, as documented in the handoff) | Dated security-to-company link records | LPERMNO, LPERMCO, GVKEY |
| Compustat Fundamentals Annual | Annual company observations, potentially with multiple formats | GVKEY, CIK, CUSIP |
| Revelio company mapping | Candidate company mappings retrieved using historical tickers | RCID, GVKEY, CIK, CUSIP |

**PERMNO** identifies a CRSP security. 

**PERMCO** identifies a CRSP company.

Multiple securities can belong to one company. 

**GVKEY** identifies a Compustat company.

**CIK** identifies an SEC filer. 

**RCID** identifies a Revelio entity. 

**CUSIP** can refer to a security issue.

**ticker** is not a reliable company identifier.


**Historical validity matters.** A candidate identifier match alone does not establish
that two records refer to the same entity during the relevant membership period.
CCM link dates, corporate changes, and parent/subsidiary relationships require review.
The Revelio extract is a candidate mapping; its parent hierarchy is not assumed to be
historically valid throughout the study period.

**Data handling.** Keep licensed records local. Display aggregate diagnostics in this
Part, preserve source files, and do not export results. Before any future GitHub upload,
the researcher must inspect and clear proprietary record-level notebook outputs.

In [188]:
from pathlib import Path

import pandas as pd
from IPython.display import display

study_start = pd.Timestamp("2018-01-01")
study_end = pd.Timestamp("2025-12-31")

# Support execution from the project root, code/, or code/notebooks/.
for project_root in [Path("."), Path(".."), Path("../..")]:
    if (project_root / "data" / "raw").is_dir():
        break
else:
    raise FileNotFoundError("Run from the project root, code/, or code/notebooks/.")

print(f"Study period: {study_start.date()} to {study_end.date()}")
print(f"pandas version: {pd.__version__}")

Study period: 2018-01-01 to 2025-12-31
pandas version: 3.0.1


## Part 1 Data Inventory and Initial Quality Checks

### 1.1 Input inventory




In [189]:
input_paths = {
    "crsp": project_root / "data/raw/crsp/crsp_sp500_20261022.csv",
    "ccm": project_root / "data/raw/ccm/ccm_link_history_20261023.csv",
    "compustat": project_root / "data/raw/compustat/Compustat Fundamentals Annual.csv",
    "revelio": project_root / "data/raw/revelio/revelio_company_mapping_20261023.csv",
}

intermediate_paths = {
    "membership_intervals": project_root / "data/clean/sp500_membership_intervals.csv",
    "permno_query_aid": project_root / "data/clean/sp500_permno_for_ccm.txt",
    "ticker_query_aid": project_root / "data/clean/sp500_tickers_for_revelio.txt",
}

inventory_rows = []
for name, path in {**input_paths, **intermediate_paths}.items():
    inventory_rows.append({
        "input": name,
        "relative_path": path.relative_to(project_root).as_posix(),
        "exists": path.is_file(),
        "size_mb": path.stat().st_size / 1_000_000 if path.is_file() else pd.NA,
    })

inventory = pd.DataFrame(inventory_rows)
display(inventory.round({"size_mb": 2}))

if not inventory["exists"].all():
    raise FileNotFoundError("An expected input is missing. Review the inventory above.")

,input,relative_path,exists,size_mb
0,crsp,data/raw/crsp/crsp_sp500_20261022.csv,True,305.54
1,ccm,data/raw/ccm/ccm_link_history_20261023.csv,True,0.03
2,compustat,data/raw/compustat/Compustat Fundamentals Annu...,True,10.85
3,revelio,data/raw/revelio/revelio_company_mapping_20261...,True,0.25
4,membership_intervals,data/clean/sp500_membership_intervals.csv,True,0.03
5,permno_query_aid,data/clean/sp500_permno_for_ccm.txt,True,0.00
6,ticker_query_aid,data/clean/sp500_tickers_for_revelio.txt,True,0.00


### 1.2 Source schemas and reading choices

Read each CSV header before loading data. The CRSP file contains many trading variables;
only nine membership and identifier fields are needed for this initial inspection.
All columns in the other three raw files are read.

Identifiers such as **GVKEY, CIK, and CUSIP are strings**, not quantities. String storage
preserves leading zeros and alphanumeric characters. Dates and numeric variables are
also initially retained as strings so that source encodings remain visible. Temporary
date conversions below are used only for diagnostics.

The reading rule treats an empty CSV field as missing, but disables pandas' broader
default missing-token list. A literal `NA` may be a ticker, so it must not be silently
converted to a missing identifier. Other possible missing tokens are counted separately.
Column names are lowercased in memory; source files are unchanged.

In [190]:
source_columns = {}
for name, path in input_paths.items():
    source_columns[name] = pd.read_csv(path, nrows=0).columns.tolist()
    print(f"{name}: {len(source_columns[name])} source columns")
    print(", ".join(source_columns[name]))
    print()

crsp: 42 source columns
INDNO, MbrStartDt, MbrEndDt, PERMNO, HdrCUSIP, PERMCO, SICCD, CUSIP, Ticker, ShareType, SecurityType, SecuritySubType, USIncFlg, IssuerType, PrimaryExch, ConditionalType, TradingStatusFlg, DlyCalDt, DlyPrc, DlyPrcFlg, DlyCap, DlyCapFlg, DlyRet, DlyRetx, DlyRetMissFlg, DlyRetDurFlg, DlyOrdDivAmt, DlyNonOrdDivAmt, DlyFacPrc, DlyDistRetFlg, DlyVol, DlyClose, DlyLow, DlyHigh, DlyBid, DlyAsk, DlyOpen, DlyNumTrd, DlyMMCnt, DlyCumFacPr, DlyCumFacShr, ShrOut

ccm: 8 source columns
GVKEY, LINKPRIM, LIID, LINKTYPE, LPERMNO, LPERMCO, LINKDT, LINKENDDT

compustat: 15 source columns
costat, curcd, datafmt, indfmt, consol, tic, datadate, gvkey, conm, cusip, cik, naics, sic, fyear, emp

revelio: 16 source columns
ticker, rcid, company, factset_entity_id, isin, cusip, url, naics_code, cik, lei, linkedin_url, child_rcid, child_company, ultimate_parent_rcid, ultimate_parent_company_name, gvkey



In [191]:
crsp_columns = [
    "INDNO", "MbrStartDt", "MbrEndDt", "PERMNO", "PERMCO",
    "CUSIP", "HdrCUSIP", "Ticker", "DlyCalDt",
]

crsp = pd.read_csv(
    input_paths["crsp"], usecols=crsp_columns,
    dtype="string", keep_default_na=False, na_values=[""],
)
ccm = pd.read_csv(
    input_paths["ccm"], dtype="string", keep_default_na=False, na_values=[""],
)
compustat = pd.read_csv(
    input_paths["compustat"], dtype="string", keep_default_na=False, na_values=[""],
)
revelio = pd.read_csv(
    input_paths["revelio"], dtype="string", keep_default_na=False, na_values=[""],
)

datasets = {
    "crsp": crsp,
    "ccm": ccm,
    "compustat": compustat,
    "revelio": revelio,
}
for frame in datasets.values():
    frame.columns = frame.columns.str.lower()

shape_rows = []
for name, frame in datasets.items():
    shape_rows.append({
        "dataset": name,
        "rows": len(frame),
        "source_columns": len(source_columns[name]),
        "loaded_columns": len(frame.columns),
    })

display(pd.DataFrame(shape_rows))

,dataset,rows,source_columns,loaded_columns
0,crsp,1013723,42,9
1,ccm,788,8,8
2,compustat,107179,15,15
3,revelio,1124,16,16


### 1.3 Identifier counts

Counts below use nonmissing, nonblank values after trimming surrounding whitespace
**for diagnostics only**. Source values in the loaded tables are not replaced. Leading
zeros are preserved, and no cross-source identifier normalization is performed.
Multiple rows per identifier may be expected given each source's unit of observation.

In [192]:
identifier_columns = {
    "crsp": ["indno", "permno", "permco", "cusip", "hdrcusip", "ticker"],
    "ccm": ["lpermno", "lpermco", "gvkey"],
    "compustat": ["gvkey", "cik", "cusip", "tic"],
    "revelio": ["rcid", "gvkey", "cik", "cusip", "ticker"],
}

identifier_rows = []
for name, columns in identifier_columns.items():
    for column in columns:
        values = datasets[name][column].str.strip()
        present = values.notna() & values.ne("")
        identifier_rows.append({
            "dataset": name,
            "identifier": column,
            "rows_with_value": int(present.sum()),
            "rows_missing_or_blank": int((~present).sum()),
            "unique_values": values.loc[present].nunique(),
        })

display(pd.DataFrame(identifier_rows))

,dataset,identifier,rows_with_value,rows_missing_or_blank,unique_values
0,crsp,indno,1013723,0,1
1,crsp,permno,1013723,0,664
2,crsp,permco,1013723,0,657
3,crsp,cusip,1013692,31,705
4,crsp,hdrcusip,1013723,0,664
5,crsp,ticker,1013692,31,697
6,ccm,lpermno,788,0,664
7,ccm,lpermco,788,0,657
8,ccm,gvkey,788,0,674
9,compustat,gvkey,107179,0,18502


### 1.4 Missing values and possible missing encodings

For each loaded column, report empty-field missing values separately from whitespace-only
strings. The percentage combines those two categories and uses **all rows of that
source** as its denominator. CRSP diagnostics cover only the nine loaded fields.

The final column flags strings that can sometimes encode missing data. These are
**candidates for review, not confirmed missing values**, and are not recoded. This list
is not exhaustive. `E` is deliberately excluded: it represents an open-ended CCM link.

In [193]:
possible_missing_tokens = [".", "NA", "N/A", "NULL", "null", "NaN", "nan", "None"]

for name, frame in datasets.items():
    missing_rows = []
    for column in frame.columns:
        values = frame[column]
        stripped = values.str.strip()
        empty_fields = int(values.isna().sum())
        whitespace_only = int(stripped.eq("").sum())
        missing_rows.append({
            "column": column,
            "empty_fields": empty_fields,
            "whitespace_only": whitespace_only,
            "missing_or_blank_pct": (
                100 * (empty_fields + whitespace_only) / len(frame)
                if len(frame) else float("nan")
            ),
            "possible_token_rows": int(stripped.isin(possible_missing_tokens).sum()),
        })
    print(name)
    display(pd.DataFrame(missing_rows).round({"missing_or_blank_pct": 2}))

crsp


,column,empty_fields,whitespace_only,missing_or_blank_pct,possible_token_rows
0,indno,0,0,0.0,0
1,mbrstartdt,0,0,0.0,0
2,mbrenddt,0,0,0.0,0
3,permno,0,0,0.0,0
4,hdrcusip,0,0,0.0,0
5,permco,0,0,0.0,0
6,cusip,31,0,0.0,0
7,ticker,31,0,0.0,0
8,dlycaldt,0,0,0.0,0


ccm


,column,empty_fields,whitespace_only,missing_or_blank_pct,possible_token_rows
0,gvkey,0,0,0.0,0
1,linkprim,0,0,0.0,0
2,liid,0,0,0.0,0
3,linktype,0,0,0.0,0
4,lpermno,0,0,0.0,0
5,lpermco,0,0,0.0,0
6,linkdt,0,0,0.0,0
7,linkenddt,0,0,0.0,0


compustat


,column,empty_fields,whitespace_only,missing_or_blank_pct,possible_token_rows
0,costat,0,0,0.00,0
1,curcd,0,0,0.00,0
2,datafmt,0,0,0.00,0
3,indfmt,0,0,0.00,0
4,consol,0,0,0.00,0
5,tic,24,0,0.02,6
6,datadate,0,0,0.00,0
7,gvkey,0,0,0.00,0
8,conm,0,0,0.00,0
9,cusip,24,0,0.02,0


revelio


,column,empty_fields,whitespace_only,missing_or_blank_pct,possible_token_rows
0,ticker,0,0,0.00,0
1,rcid,0,0,0.00,0
2,company,0,0,0.00,0
3,factset_entity_id,0,0,0.00,0
4,isin,0,0,0.00,0
5,cusip,2,0,0.18,0
6,url,10,0,0.89,0
7,naics_code,0,0,0.00,0
8,cik,426,0,37.90,0
9,lei,226,0,20.11,0


### 1.5 Basic duplicate diagnostics

An exact duplicate means all **loaded** column values repeat. For CRSP, this is a
nine-column projection, not a check of equality across all 42 source columns.

A repeated key is a different concept. The table reports both extra rows beyond the
first occurrence and all rows involved in repeated groups. Rows with any missing or
blank key component are counted separately and excluded from the key-duplicate check.
No rows are dropped from the input tables.

Compustat `GVKEY + FYEAR` and `GVKEY + DATADATE` need not be unique because formats and
other dimensions can differ. Even the expanded key below is a diagnostic, not an approved
record-selection rule. A repeated Revelio GVKEY or ticker does not by itself prove an
incorrect mapping.

In [194]:
exact_duplicate_rows = []
for name, frame in datasets.items():
    exact_duplicate_rows.append({
        "dataset": name,
        "columns_checked": len(frame.columns),
        "extra_identical_rows": int(frame.duplicated().sum()),
        "all_rows_in_identical_groups": int(frame.duplicated(keep=False).sum()),
    })

display(pd.DataFrame(exact_duplicate_rows))

,dataset,columns_checked,extra_identical_rows,all_rows_in_identical_groups
0,crsp,9,0,0
1,ccm,8,0,0
2,compustat,15,0,0
3,revelio,16,0,0


In [195]:
key_checks = [
    ("crsp", ["permno", "dlycaldt"]),
    ("ccm", ["lpermno", "gvkey", "liid", "linkdt", "linkenddt", "linkprim", "linktype"]),
    ("compustat", ["gvkey", "fyear"]),
    ("compustat", ["gvkey", "datadate"]),
    ("compustat", ["gvkey", "datadate", "indfmt", "datafmt", "consol", "curcd"]),
    ("revelio", ["rcid"]),
    ("revelio", ["gvkey"]),
    ("revelio", ["ticker"]),
]

key_rows = []
for name, columns in key_checks:
    keys = datasets[name][columns].copy()
    for column in columns:
        keys[column] = keys[column].str.strip()
    incomplete = keys.isna().any(axis=1) | keys.eq("").any(axis=1)
    complete_keys = keys.loc[~incomplete]
    key_rows.append({
        "dataset": name,
        "key": " + ".join(columns),
        "rows_with_incomplete_key": int(incomplete.sum()),
        "extra_rows_with_repeated_key": int(complete_keys.duplicated().sum()),
        "all_rows_with_repeated_key": int(complete_keys.duplicated(keep=False).sum()),
    })

display(pd.DataFrame(key_rows))

,dataset,key,rows_with_incomplete_key,extra_rows_with_repeated_key,all_rows_with_repeated_key
0,crsp,permno + dlycaldt,0,0,0
1,ccm,lpermno + gvkey + liid + linkdt + linkenddt + ...,0,0,0
2,compustat,gvkey + fyear,0,8179,16358
3,compustat,gvkey + datadate,0,8176,16352
4,compustat,gvkey + datadate + indfmt + datafmt + consol +...,0,0,0
5,revelio,rcid,0,0,0
6,revelio,gvkey,214,0,0
7,revelio,ticker,0,496,781


### 1.6 Date coverage and open-ended links

Parse ISO dates into temporary Series; retain the original date columns in the input
tables. Report missing/blank values, open-ended markers, and other unparseable values
separately. Minimum and maximum dates refer only to successfully parsed dates.

For CCM `LINKENDDT`, `E` means **open-ended**, not unknown historical coverage. No
replacement end date is applied in this Part. A proposed later overlap check would use
`study_end` as the effective analysis end for an `E` link while retaining its original
marker. That proposal requires review before implementation.

Dates outside the study window are not automatically errors: membership and link
histories can legitimately start earlier. Compustat `DATADATE` is a financial reporting
date, not a substitute for `FYEAR`. Daily coverage follows observed trading dates.

In [196]:
date_columns = {
    "crsp": ["dlycaldt", "mbrstartdt", "mbrenddt"],
    "ccm": ["linkdt", "linkenddt"],
    "compustat": ["datadate"],
}

date_rows = []
for name, columns in date_columns.items():
    for column in columns:
        values = datasets[name][column].str.strip()
        missing = values.isna() | values.eq("").fillna(False)
        open_ended = pd.Series(False, index=values.index)
        if name == "ccm" and column == "linkenddt":
            open_ended = values.eq("E").fillna(False)
        parsed_dates = pd.to_datetime(
            values.mask(open_ended), format="%Y-%m-%d", errors="coerce",
        )
        invalid = parsed_dates.isna() & ~missing & ~open_ended
        date_rows.append({
            "dataset": name,
            "column": column,
            "earliest_parsed_date": parsed_dates.min(),
            "latest_parsed_date": parsed_dates.max(),
            "missing_or_blank": int(missing.sum()),
            "open_ended_E": int(open_ended.sum()),
            "other_unparseable": int(invalid.sum()),
            "rows_before_study": int(parsed_dates.lt(study_start).sum()),
            "rows_after_study": int(parsed_dates.gt(study_end).sum()),
        })

display(pd.DataFrame(date_rows))

,dataset,column,earliest_parsed_date,latest_parsed_date,missing_or_blank,open_ended_E,other_unparseable,rows_before_study,rows_after_study
0,crsp,dlycaldt,2018-01-02,2025-12-31,0,0,0,0,0
1,crsp,mbrstartdt,1925-12-31,2025-12-22,0,0,0,872000,0
2,crsp,mbrenddt,2018-03-06,2025-12-31,0,0,0,0,0
3,ccm,linkdt,1949-07-01,2025-11-03,0,0,0,734,0
4,ccm,linkenddt,1962-01-01,2025-12-31,0,592,0,122,0
5,compustat,datadate,2018-01-31,2025-12-31,0,0,0,0,0


### 1.7 Source-specific aggregate checks

Review CCM link categories without filtering them. Review Compustat formats and fiscal
years without selecting an annual record. Count distinct RCID values per nonblank
Revelio ticker without matching any firms.

These diagnostics describe the downloaded extracts. In particular, Revelio coverage
cannot be inferred from ticker counts, and Compustat EMP missingness above describes
the full extract rather than the eventual historical research sample.

In [197]:
print("CRSP index identifiers")
display(crsp["indno"].value_counts(dropna=False).rename_axis("indno").to_frame("rows"))

for column in ["linktype", "linkprim"]:
    print(f"CCM: {column}")
    display(ccm[column].value_counts(dropna=False).rename_axis(column).to_frame("rows"))

for column in ["indfmt", "fyear"]:
    print(f"Compustat: {column}")
    display(
        compustat[column].value_counts(dropna=False)
        .sort_index().rename_axis(column).to_frame("rows")
    )

CRSP index identifiers


,rows
indno,
1000500,1013723


CCM: linktype


,rows
linktype,
LC,543
LU,242
LS,3


CCM: linkprim


,rows
linkprim,
P,667
C,108
J,11
N,2


Compustat: indfmt


,rows
indfmt,
FS,8176
INDL,99003


Compustat: fyear


,rows
fyear,
2017,970
2018,12440
2019,12647
2020,12966
2021,13315
2022,13612
2023,13698
2024,13913
2025,13618


In [198]:
revelio_ticker_keys = revelio[["ticker", "rcid"]].copy()
for column in revelio_ticker_keys.columns:
    revelio_ticker_keys[column] = revelio_ticker_keys[column].str.strip()

valid_ticker = (
    revelio_ticker_keys["ticker"].notna()
    & revelio_ticker_keys["ticker"].ne("")
)
valid_rcid = (
    revelio_ticker_keys["rcid"].notna()
    & revelio_ticker_keys["rcid"].ne("")
)
rcids_per_ticker = (
    revelio_ticker_keys.loc[valid_ticker & valid_rcid]
    .groupby("ticker")["rcid"].nunique()
)

display(pd.DataFrame([{
    "rows_excluded_for_missing_ticker_or_rcid": int((~(valid_ticker & valid_rcid)).sum()),
    "tickers_with_nonmissing_rcid": len(rcids_per_ticker),
    "tickers_with_multiple_rcids": int(rcids_per_ticker.gt(1).sum()),
}]))

,rows_excluded_for_missing_ticker_or_rcid,tickers_with_nonmissing_rcid,tickers_with_multiple_rcids
0,0,628,285


### Findings

- The CRSP extract contains 1,013,723 daily observations for 664 securities (PERMNO)
  and 657 companies (PERMCO), all under index identifier 1000500. Daily dates run from
  January 2, 2018 to December 31, 2025, with no repeated PERMNO-date keys. These are
  daily-data counts; membership-interval counts remain to be checked in Part 2.
- CCM contains 788 link records for 664 distinct LPERMNO. Of the 788 end-date values,
  592 are open-ended `E` markers. Equal security counts across CRSP and CCM do not
  establish identifier coverage or historically valid matching.
- Compustat contains 107,179 observations for 18,502 GVKEY, including 8,176 FS-format
  records. There are 8,176 extra rows beyond the first per GVKEY-date key, while the
  expanded key including format, consolidation, and currency has no repeats. EMP is
  blank in 46,878 of 107,179 rows (43.74%); this is extract-wide missingness, not
  research-sample missingness. No annual record-selection rule has been applied.
- Compustat reporting dates span January 31, 2018 to December 31, 2025, but 970 rows
  have fiscal year 2017. Reporting-date coverage and fiscal-year coverage must be
  distinguished when defining the subsequent annual panel.
- Revelio contains 1,124 distinct RCID, of which 910 have a nonblank GVKEY. Of 628
  distinct tickers, 285 correspond to multiple RCID. These results support further
  identifier validation; they do not measure historical-sample Revelio coverage.

## Part 2 - Historical S&P 500 Membership

One row represents a recorded membership interval for a security, not a company-year.
A PERMCO can have multiple PERMNO, so security and company counts need not agree.

Run **Part 0** setup before this Part. All diagnostics are computed from the file.
No records are removed, corrected, or exported. Detailed tables are for local review.

### 2.1 Membership structure

Retain the five source columns as strings with lowercase names. 

Add a row identifier for tracing exceptions; it numbers data rows from one, excluding the CSV header.
Exact duplicate counts use the five source columns, excluding this new identifier.

In [199]:
membership_path = project_root / "data/clean/sp500_membership_intervals.csv"
membership = pd.read_csv(
    membership_path, dtype="string", keep_default_na=False, na_values=[""],
)
membership.columns = membership.columns.str.lower()

membership_columns = ["indno", "mbrstartdt", "mbrenddt", "permno", "permco"]
missing_columns = set(membership_columns) - set(membership.columns)
if missing_columns:
    raise ValueError(f"Missing membership columns: {sorted(missing_columns)}")

membership["membership_row"] = range(1, len(membership) + 1)

# Trim identifiers only in a diagnostic copy; preserve the source columns.
membership_identifiers = membership[["indno", "permno", "permco"]].copy()
for column in membership_identifiers.columns:
    membership_identifiers[column] = (
        membership_identifiers[column].str.strip().replace("", pd.NA)
    )

membership["missing_identifier"] = membership_identifiers.isna().any(axis=1)
membership["unexpected_index"] = (
    membership_identifiers["indno"].ne("1000500").fillna(False)
)
membership["exact_duplicate"] = membership.duplicated(
    subset=membership_columns, keep=False,
)

In [200]:
membership_summary = pd.Series({
    "membership_interval_records": len(membership),
    "distinct_source_records": int(
        (~membership.duplicated(subset=membership_columns)).sum()
    ),
    "unique_index_identifiers": membership_identifiers["indno"].nunique(),
    "distinct_permno": membership_identifiers["permno"].nunique(),
    "distinct_permco": membership_identifiers["permco"].nunique(),
    "rows_with_missing_identifiers": int(membership["missing_identifier"].sum()),
    "rows_in_exact_duplicate_groups": int(membership["exact_duplicate"].sum()),
}, name="count")
display(membership_summary.to_frame())
display(
    membership_identifiers["indno"].value_counts(dropna=False)
    .rename_axis("indno").to_frame("interval_records")
)

,count
membership_interval_records,666
distinct_source_records,666
unique_index_identifiers,1
distinct_permno,664
distinct_permco,657
rows_with_missing_identifiers,0
rows_in_exact_duplicate_groups,0


,interval_records
indno,
1000500,666


### 2.2 Repeated membership

Identify every nonmissing PERMNO with more than one record. Repeated records may
represent separate membership episodes or duplicate/overlapping intervals; inspect
the dates before interpreting them as re-entry. Retain every original row.

Record-level previews are limited to 30 rows to keep saved outputs compact. The full
DataFrames remain available locally; a printed total identifies any truncated preview.

In [201]:
intervals_per_permno = membership_identifiers.groupby("permno").size()
repeated_permnos = intervals_per_permno.loc[intervals_per_permno.gt(1)].index
membership["repeated_permno"] = membership_identifiers["permno"].isin(repeated_permnos)

review_columns = ["membership_row", "indno", "permno", "permco", "mbrstartdt", "mbrenddt"]
repeated_membership = membership.loc[
    membership["repeated_permno"], review_columns,
].sort_values(["permno", "mbrstartdt", "mbrenddt"])

print(f"PERMNO with multiple interval records: {len(repeated_permnos)}")
print(f"Repeated-membership records: {len(repeated_membership)}; preview limit: 30")
display(repeated_membership.head(30))

PERMNO with multiple interval records: 2
Repeated-membership records: 4; preview limit: 30


,membership_row,indno,permno,permco,mbrstartdt,mbrenddt
41,42,1000500,13688,21356,1925-12-31,2019-01-17
604,605,1000500,13688,21356,2022-10-03,2025-12-31
130,131,1000500,24328,20662,2008-12-19,2018-11-12
605,606,1000500,24328,20662,2022-10-03,2025-12-31


### 2.3 Date validation

Parse dates into new columns, leaving `mbrstartdt` and `mbrenddt` unchanged. Separate
missing/blank dates from nonblank values that cannot be parsed. Reversed intervals
have a parsed start later than the parsed end and are not eligible for overlap tests.

Flag starts on or before the previously noted date `1925-12-31` for interpretation.
This is a review flag, not a correction rule or a claim that these dates are invalid.
Also inspect the ten earliest distinct parsed start dates for other unusual values.

In [202]:
membership_date_rows = []
for source_column, parsed_column in [
    ("mbrstartdt", "membership_start"),
    ("mbrenddt", "membership_end"),
]:
    source_dates = membership[source_column].str.strip()
    missing_dates = source_dates.isna() | source_dates.eq("").fillna(False)
    membership[parsed_column] = pd.to_datetime(
        source_dates, format="%Y-%m-%d", errors="coerce",
    )
    invalid_dates = membership[parsed_column].isna() & ~missing_dates
    membership[f"{parsed_column}_missing"] = missing_dates
    membership[f"{parsed_column}_invalid"] = invalid_dates
    membership_date_rows.append({
        "column": source_column,
        "missing_or_blank": int(missing_dates.sum()),
        "nonblank_unparseable": int(invalid_dates.sum()),
        "earliest_parsed": membership[parsed_column].min(),
        "latest_parsed": membership[parsed_column].max(),
    })

membership["reversed_interval"] = (
    membership["membership_start"] > membership["membership_end"]
)
membership["valid_dates"] = (
    membership[["membership_start", "membership_end"]].notna().all(axis=1)
    & ~membership["reversed_interval"]
)
membership["early_start_for_review"] = (
    membership["membership_start"] <= pd.Timestamp("1925-12-31")
)

display(pd.DataFrame(membership_date_rows))
print(f"Reversed intervals: {int(membership['reversed_interval'].sum())}")
display(
    membership["membership_start"].value_counts().sort_index().head(10)
    .rename_axis("earliest_start_dates").to_frame("interval_records")
)

,column,missing_or_blank,nonblank_unparseable,earliest_parsed,latest_parsed
0,mbrstartdt,0,0,1925-12-31,2025-12-22
1,mbrenddt,0,0,2018-03-06,2025-12-31


Reversed intervals: 0


,interval_records
earliest_start_dates,
1925-12-31,10
1944-06-07,2
1948-07-07,1
1949-07-06,1
1951-09-06,1
1957-03-01,34
1957-07-10,1
1959-03-25,1
1959-04-01,1


### 2.4 Study-window boundaries

Use inclusive calendar-date boundaries: an interval overlaps the study period when
its start is on or before `study_end` and its end is on or after `study_start`.
Intervals with missing, invalid, or reversed dates remain unassessable.

For valid overlapping intervals, `effective_start` is the later of the historical
start and `study_start`; `effective_end` is the earlier of the historical end and
`study_end`. Other records retain missing effective dates and remain in the table.
These columns describe observation within the study window, not revised membership.
An original end date equal to `2025-12-31` does not establish an actual index deletion.
Counts below are interval-record counts; the boundary flags may overlap.

In [203]:
membership["overlaps_study"] = (
    membership["valid_dates"]
    & membership["membership_start"].le(study_end)
    & membership["membership_end"].ge(study_start)
)
membership["outside_study"] = membership["valid_dates"] & ~membership["overlaps_study"]
membership["ends_at_study_boundary"] = membership["membership_end"].eq(study_end)

membership["effective_start"] = (
    membership["membership_start"].clip(lower=study_start)
    .where(membership["overlaps_study"])
)
membership["effective_end"] = (
    membership["membership_end"].clip(upper=study_end)
    .where(membership["overlaps_study"])
)

window_summary = pd.Series({
    "total_interval_records": len(membership),
    "overlapping_study_window": int(membership["overlaps_study"].sum()),
    "entirely_outside_study_window": int(membership["outside_study"].sum()),
    "unassessable_dates": int((~membership["valid_dates"]).sum()),
    "overlapping_intervals_starting_before_window": int((
        membership["overlaps_study"] & membership["membership_start"].lt(study_start)
    ).sum()),
    "overlapping_intervals_ending_after_window": int((
        membership["overlaps_study"] & membership["membership_end"].gt(study_end)
    ).sum()),
    "original_end_equals_study_end": int(membership["ends_at_study_boundary"].sum()),
}, name="interval_records")
display(window_summary.to_frame())

,interval_records
total_interval_records,666
overlapping_study_window,666
entirely_outside_study_window,0
unassessable_dates,0
overlapping_intervals_starting_before_window,504
overlapping_intervals_ending_after_window,0
original_end_equals_study_end,503


### 2.5 Overlapping intervals for the same security

Compare all distinct pairs of valid intervals within each repeated PERMNO. This small
within-table comparison also detects nested intervals; comparing only adjacent end
dates could miss them. A shared endpoint counts as overlap under inclusive boundaries.
Pairs use original historical dates, not effective dates. Exact duplicate intervals
can also appear here. Missing PERMNO and invalid/reversed intervals are excluded only
from this diagnostic and remain in `membership` for review.

In [204]:
comparable_membership = membership.loc[
    membership["repeated_permno"] & membership["valid_dates"],
    ["membership_row", "membership_start", "membership_end"],
].copy()
comparable_membership["permno"] = membership_identifiers.loc[
    comparable_membership.index, "permno",
]

interval_pairs = comparable_membership.merge(
    comparable_membership, on="permno", suffixes=("_a", "_b"),
)
# Keep each pair once, excluding a record paired with itself.
interval_pairs = interval_pairs.loc[
    interval_pairs["membership_row_a"] < interval_pairs["membership_row_b"]
].copy()
overlapping_intervals = interval_pairs.loc[
    interval_pairs["membership_start_a"].le(interval_pairs["membership_end_b"])
    & interval_pairs["membership_start_b"].le(interval_pairs["membership_end_a"])
].copy()

membership["overlapping_interval"] = (
    membership["membership_row"].isin(overlapping_intervals["membership_row_a"])
    | membership["membership_row"].isin(overlapping_intervals["membership_row_b"])
)
print(f"Overlapping interval pairs: {len(overlapping_intervals)}; preview limit: 30")
print(f"PERMNO with overlapping intervals: {overlapping_intervals['permno'].nunique()}")
display(overlapping_intervals.head(30))

Overlapping interval pairs: 0; preview limit: 30
PERMNO with overlapping intervals: 0


,membership_row_a,membership_start_a,membership_end_a,permno,membership_row_b,membership_start_b,membership_end_b


### 2.6 Exceptions for local review

Combine review flags into one table without removing any membership records. An
exception flag requests interpretation; it does not by itself imply an erroneous row.
Sample-end dates are reported separately in the boundary summary rather than labeling
all such records as errors. The combined table includes original and effective dates.
Review each flag's total and the preview; all flagged rows remain in
`membership_exceptions` if more than 30 records require inspection.

In [205]:
review_flags = [
    "missing_identifier", "unexpected_index", "exact_duplicate", "repeated_permno",
    "membership_start_missing", "membership_start_invalid",
    "membership_end_missing", "membership_end_invalid",
    "reversed_interval", "outside_study", "early_start_for_review",
    "overlapping_interval",
]

display(membership[review_flags].sum().rename("flagged_records").to_frame())

membership["review_reason"] = ""
for flag in review_flags:
    flagged = membership[flag]
    membership.loc[flagged, "review_reason"] = (
        membership.loc[flagged, "review_reason"] + flag + "; "
    )
membership["review_reason"] = membership["review_reason"].str.rstrip("; ")

exception_columns = review_columns + ["effective_start", "effective_end", "review_reason"]
membership_exceptions = membership.loc[
    membership[review_flags].any(axis=1), exception_columns,
].sort_values(["permno", "mbrstartdt", "membership_row"])

print(f"Records requiring review: {len(membership_exceptions)}; preview limit: 30")
with pd.option_context("display.max_columns", None, "display.max_colwidth", 120):
    display(membership_exceptions.head(30))

,flagged_records
missing_identifier,0
unexpected_index,0
exact_duplicate,0
repeated_permno,4
membership_start_missing,0
membership_start_invalid,0
membership_end_missing,0
membership_end_invalid,0
reversed_interval,0
outside_study,0


Records requiring review: 13; preview limit: 30


,membership_row,indno,permno,permco,mbrstartdt,mbrenddt,effective_start,effective_end,review_reason
3,4,1000500,10145,22168,1925-12-31,2025-12-31,2018-01-01,2025-12-31,early_start_for_review
9,10,1000500,11404,20499,1925-12-31,2025-12-31,2018-01-01,2025-12-31,early_start_for_review
13,14,1000500,11674,20574,1925-12-31,2025-12-31,2018-01-01,2025-12-31,early_start_for_review
15,16,1000500,11850,20678,1925-12-31,2025-12-31,2018-01-01,2025-12-31,early_start_for_review
41,42,1000500,13688,21356,1925-12-31,2019-01-17,2018-01-01,2019-01-17,repeated_permno; early_start_for_review
604,605,1000500,13688,21356,2022-10-03,2025-12-31,2022-10-03,2025-12-31,repeated_permno
57,58,1000500,14541,20440,1925-12-31,2025-12-31,2018-01-01,2025-12-31,early_start_for_review
65,66,1000500,15069,21795,1925-12-31,2024-11-25,2018-01-01,2024-11-25,early_start_for_review
72,73,1000500,15720,21543,1925-12-31,2025-12-31,2018-01-01,2025-12-31,early_start_for_review
76,77,1000500,16432,20834,1925-12-31,2019-02-26,2018-01-01,2019-02-26,early_start_for_review


### Findings

- The table contains 666 distinct membership intervals for 664 PERMNO and 657 PERMCO, all under index identifier 1000500. No missing identifiers or exact duplicate records were found.
- Two securities each have two non-overlapping intervals. These records are consistent with separate membership episodes and were retained without collapsing them.
- All 666 intervals overlap the 2018-2025 study window. Dates are complete and parseable, with no reversed intervals or overlapping periods for the same security. Effective starts were limited to January 1, 2018 for 504 intervals; original dates were preserved.
- Ten intervals begin on December 31, 1925 and require historical interpretation. Another boundary issue concerns the 503 intervals ending on December 31, 2025: this sample-end date should not be treated as a confirmed index deletion.
- The review table contains 13 distinct records: 10 with early starts and four belonging to repeated-membership securities, with one record in both groups. All records remain available for subsequent CCM linking; no dates or membership episodes were corrected or removed.

## Part 3 - CCM Historical Linking

Connect membership intervals to candidate Compustat identifiers using CCM link history.
Run Parts 0-2 first. Work on copies of `membership` and `ccm`; preserve all source
values and prior outputs. One candidate row represents a membership interval paired
with a CCM record, not a unique company.

Compare identifier-only coverage, date-overlapping coverage, and date-overlapping
P/C coverage. These are diagnostic candidate sets, not a finalized crosswalk. Retain
all LINKTYPE categories for comparison; do not silently restrict LC, LU, or LS, or
promote J/N links into the P/C set. Record previews are limited to 30 rows; the named
DataFrames retain all diagnostic records locally.

### CCM Primary Link Markers (`LINKPRIM`)

| LINKPRIM | Full Name | Meaning | Treatment in Our Crosswalk |
|---|---|---|---|
| **P** | Primary | Primary security designated by Compustat. | Include as a primary-link candidate. |
| **C** | CRSP-assigned Primary | Primary link assigned by CRSP to maintain a consistent historical primary-security mapping. | Include as a primary-link candidate. |
| **J** | Joiner | Secondary security linked to a Compustat company (GVKEY), but not designated as its primary security. | Retain for review; may be valid for firm identification but is excluded from the initial P/C primary-link filter. |
| **N** | Non-primary Link | A non-primary or nonstandard link designation requiring further examination. | Retain for review; do not automatically include as a primary link. |

**Notes:**
- `LINKPRIM` describes a security's primary-link status, **not whether the company match is necessarily correct or incorrect**.
- `P` and `C` are the standard primary-link candidates.
- `J` can still represent a valid security-to-company relationship, particularly when a company has multiple share classes.
- `LINKTYPE` (e.g., `LC`, `LU`, `LS`) is a separate indicator of link type and quality.
- All candidate links must also satisfy the relevant historical date-overlap requirements.


### 3.1 Prepare identifiers and link dates

Trim whitespace in separate identifier keys without numeric conversion. Keep original
`linkdt` and `linkenddt`. For `E`, set only `link_end_effective` to `study_end`:
this bounds the analysis, not the actual link history. Other missing/unparseable date
values remain unresolved. An open-ended link starting after the study window is not
reversed, although it cannot overlap the sample.

In [206]:
ccm_history = ccm.copy()
ccm_history["ccm_row"] = range(1, len(ccm_history) + 1)

for source_column, key_column in [
    ("lpermno", "permno_key"), ("lpermco", "ccm_permco_key"), ("gvkey", "gvkey_key"),
]:
    ccm_history[key_column] = (
        ccm_history[source_column].str.strip().replace("", pd.NA)
    )
ccm_history["linkprim_key"] = ccm_history["linkprim"].str.strip()
ccm_history["linktype_key"] = ccm_history["linktype"].str.strip()
ccm_history["duplicate_ccm_record"] = ccm_history.duplicated(
    subset=list(ccm.columns), keep=False,
)

link_start_text = ccm_history["linkdt"].str.strip()
link_end_text = ccm_history["linkenddt"].str.strip()
ccm_history["open_ended"] = link_end_text.eq("E").fillna(False)
ccm_history["link_start"] = pd.to_datetime(
    link_start_text, format="%Y-%m-%d", errors="coerce",
)
ccm_history["link_end_parsed"] = pd.to_datetime(
    link_end_text.mask(ccm_history["open_ended"]),
    format="%Y-%m-%d", errors="coerce",
)
ccm_history["link_end_effective"] = ccm_history["link_end_parsed"].where(
    ~ccm_history["open_ended"], study_end,
)

In [207]:
ccm_history["missing_link_start"] = link_start_text.isna() | link_start_text.eq("").fillna(False)
ccm_history["missing_link_end"] = link_end_text.isna() | link_end_text.eq("").fillna(False)
ccm_history["invalid_link_start"] = (
    ccm_history["link_start"].isna() & ~ccm_history["missing_link_start"]
)
ccm_history["invalid_link_end"] = (
    ccm_history["link_end_parsed"].isna()
    & ~ccm_history["missing_link_end"] & ~ccm_history["open_ended"]
)
ccm_history["reversed_link"] = (
    ccm_history["link_start"] > ccm_history["link_end_parsed"]
)
ccm_history["valid_link_dates"] = (
    ccm_history["link_start"].notna()
    & ccm_history["link_end_effective"].notna()
    & ~ccm_history["reversed_link"]
)

link_date_flags = [
    "open_ended", "missing_link_start", "missing_link_end",
    "invalid_link_start", "invalid_link_end", "reversed_link", "valid_link_dates",
]
display(ccm_history[link_date_flags].sum().rename("ccm_records").to_frame())
display(pd.crosstab(
    ccm_history["linktype_key"], ccm_history["linkprim_key"], dropna=False,
))

,ccm_records
open_ended,592
missing_link_start,0
missing_link_end,0
invalid_link_start,0
invalid_link_end,0
reversed_link,0
valid_link_dates,788


linkprim_key,C,J,N,P
linktype_key,,,,
LC,72,11,2,458
LS,1,0,0,2
LU,35,0,0,207


### 3.2 Candidate links by PERMNO

Left-join every membership interval to CCM on PERMNO. Multiple records on either side
are expected: preserve `membership_row` and `ccm_row` so candidate expansion is visible.
Exclude missing CCM PERMNO only from the join to avoid pandas matching null keys;
those source records remain in the CCM exception table. Keep membership intervals
without candidates in the left-join result.

In [208]:
membership_for_ccm = membership[[
    "membership_row", "indno", "permno", "permco", "mbrstartdt", "mbrenddt",
    "effective_start", "effective_end", "overlaps_study",
]].copy()
membership_for_ccm["permno_key"] = membership_for_ccm["permno"].str.strip().replace("", pd.NA)
membership_for_ccm["member_permco_key"] = membership_for_ccm["permco"].str.strip().replace("", pd.NA)

candidate_links = membership_for_ccm.merge(
    ccm_history.loc[ccm_history["permno_key"].notna()],
    on="permno_key", how="left", indicator="candidate_status",
)
candidate_links["has_ccm_record"] = candidate_links["candidate_status"].eq("both")
candidate_links["has_gvkey"] = candidate_links["gvkey_key"].notna()

candidate_summary = pd.Series({
    "membership_interval_records": len(membership_for_ccm),
    "left_join_rows_including_unmatched": len(candidate_links),
    "membership_ccm_candidate_pairs": int(candidate_links["has_ccm_record"].sum()),
    "intervals_with_any_ccm_record": candidate_links.loc[
        candidate_links["has_ccm_record"], "membership_row",
    ].nunique(),
}, name="count")
display(candidate_summary.to_frame())

,count
membership_interval_records,666
left_join_rows_including_unmatched,792
membership_ccm_candidate_pairs,792
intervals_with_any_ccm_record,666


### 3.3 Historical overlap and P/C links

Within the study window, calculate:
`overlap_start = max(effective_start, link_start)` and
`overlap_end = min(effective_end, link_end_effective)`.
Require valid dates and `overlap_start <= overlap_end`, including shared endpoints.
Missing dates must not be skipped when taking the maximum or minimum.

`date_valid_links` additionally requires a nonmissing GVKEY. `primary_links` is its
P/C subset. The name describes the LINKPRIM filter, not final researcher verification.
A link with any overlap may cover only part of a membership interval. For disjoint
valid intervals, `boundary_separation_days` measures the distance between endpoints;
a value of one is a boundary discrepancy for review, not permission to change dates.

In [209]:
candidate_links["overlap_start"] = candidate_links[
    ["effective_start", "link_start"]
].max(axis=1, skipna=False)
candidate_links["overlap_end"] = candidate_links[
    ["effective_end", "link_end_effective"]
].min(axis=1, skipna=False)

assessable_dates = (
    candidate_links["has_ccm_record"]
    & candidate_links["overlaps_study"]
    & candidate_links["valid_link_dates"].fillna(False)
)
candidate_links["date_overlap"] = (
    assessable_dates & candidate_links["overlap_start"].le(candidate_links["overlap_end"])
)
candidate_links["boundary_separation_days"] = (
    candidate_links["overlap_start"] - candidate_links["overlap_end"]
).dt.days.where(assessable_dates & ~candidate_links["date_overlap"])
candidate_links["is_primary"] = candidate_links["linkprim_key"].isin(["P", "C"])
candidate_links["spans_full_interval"] = (
    candidate_links["date_overlap"]
    & candidate_links["link_start"].le(candidate_links["effective_start"])
    & candidate_links["link_end_effective"].ge(candidate_links["effective_end"])
)

date_valid_links = candidate_links.loc[
    candidate_links["date_overlap"] & candidate_links["has_gvkey"]
].copy()
primary_links = date_valid_links.loc[date_valid_links["is_primary"]].copy()

### 3.4 Coverage by stage

For each stage, count unique membership intervals and unique PERMNO, not joined rows.
Denominators are all input membership records and all distinct nonmissing input PERMNO,
respectively. The GVKEY column is a count of candidate companies, not a coverage rate.
No LINKTYPE filter is applied. The final row shows the additional requirement that
one P/C record spans the entire effective interval; failure of that condition does
not prove a gap, because several links could jointly cover an interval.

In [210]:
coverage_stages = {
    "Any CCM record, dates ignored": candidate_links.loc[candidate_links["has_ccm_record"]],
    "CCM record with GVKEY, dates ignored": candidate_links.loc[
        candidate_links["has_ccm_record"] & candidate_links["has_gvkey"]
    ],
    "Date overlap with GVKEY, all LINKPRIM": date_valid_links,
    "Date overlap with GVKEY, P/C only": primary_links,
    "Single P/C link spans effective interval": primary_links.loc[
        primary_links["spans_full_interval"]
    ],
}
interval_denominator = len(membership_for_ccm)
permno_denominator = membership_for_ccm["permno_key"].nunique()
coverage_rows = []
for stage, links in coverage_stages.items():
    interval_count = links["membership_row"].nunique()
    permno_count = links["permno_key"].nunique()
    coverage_rows.append({
        "stage": stage,
        "matched_intervals": interval_count,
        "total_intervals": interval_denominator,
        "interval_pct": 100 * interval_count / interval_denominator if interval_denominator else float("nan"),
        "matched_permno": permno_count,
        "total_permno": permno_denominator,
        "permno_pct": 100 * permno_count / permno_denominator if permno_denominator else float("nan"),
        "candidate_gvkey": links["gvkey_key"].nunique(),
    })
display(pd.DataFrame(coverage_rows).round({"interval_pct": 2, "permno_pct": 2}))
display(pd.crosstab(date_valid_links["linktype_key"], date_valid_links["linkprim_key"]))

,stage,matched_intervals,total_intervals,interval_pct,matched_permno,total_permno,permno_pct,candidate_gvkey
0,"Any CCM record, dates ignored",666,666,100.00,664,664,100.00,674
1,"CCM record with GVKEY, dates ignored",666,666,100.00,664,664,100.00,674
2,"Date overlap with GVKEY, all LINKPRIM",665,666,99.85,663,664,99.85,652
3,"Date overlap with GVKEY, P/C only",659,666,98.95,657,664,98.95,652
4,Single P/C link spans effective interval,658,666,98.80,656,664,98.80,651


linkprim_key,C,J,P
linktype_key,,,
LC,7,6,435
LS,0,0,2
LU,8,0,207


### 3.5 Intervals without P/C overlap and boundary exceptions

List intervals with no date-overlapping P/C link to a nonmissing GVKEY. Attach all of
their original CCM candidates, including J/N, non-overlapping dates, and invalid dates.
Show one-day endpoint differences separately without adjusting them. The secondary
link table retains every date-overlapping non-P/C candidate, even for intervals that
also have a P/C candidate. All tables are diagnostic; none supplies a fallback match.

In [211]:
interval_link_status = membership_for_ccm.copy()
interval_link_status["has_candidate"] = interval_link_status["membership_row"].isin(
    candidate_links.loc[candidate_links["has_ccm_record"], "membership_row"]
)
interval_link_status["has_date_valid_link"] = interval_link_status["membership_row"].isin(
    date_valid_links["membership_row"]
)
interval_link_status["has_primary_link"] = interval_link_status["membership_row"].isin(
    primary_links["membership_row"]
)
unmatched_intervals = interval_link_status.loc[
    ~interval_link_status["has_primary_link"]
].copy()

link_review_columns = [
    "membership_row", "permno", "permco", "effective_start", "effective_end",
    "ccm_row", "gvkey", "lpermco", "linkprim", "linktype", "linkdt", "linkenddt",
    "date_overlap", "boundary_separation_days",
]
unmatched_candidates = candidate_links.loc[
    candidate_links["membership_row"].isin(unmatched_intervals["membership_row"]),
    link_review_columns,
].copy()
print(f"Intervals without a date-overlapping P/C link: {len(unmatched_intervals)}")
display(unmatched_intervals.head(30))
print(f"Their candidate rows: {len(unmatched_candidates)}; preview limit: 30")
with pd.option_context("display.max_columns", None):
    display(unmatched_candidates.head(30))

Intervals without a date-overlapping P/C link: 7


,membership_row,indno,permno,permco,mbrstartdt,mbrenddt,effective_start,effective_end,overlaps_study,permno_key,member_permco_key,has_candidate,has_date_valid_link,has_primary_link
51,52,1000500,13964,54433,2015-09-21,2025-12-31,2018-01-01,2025-12-31,True,13964,54433,True,True,False
58,59,1000500,14542,45483,2014-04-03,2025-12-31,2018-01-01,2025-12-31,True,14542,45483,True,True,False
74,75,1000500,15980,48779,2016-04-08,2022-06-17,2018-01-01,2022-06-17,True,15980,48779,True,True,False
471,472,1000500,90442,21287,2015-09-21,2019-03-19,2018-01-01,2019-03-19,True,90442,21287,True,True,False
499,500,1000500,92778,47011,2014-08-07,2022-04-08,2018-01-01,2022-04-08,True,92778,47011,True,True,False
510,511,1000500,17676,56329,2018-05-01,2018-05-01,2018-05-01,2018-05-01,True,17676,56329,True,False,False
538,539,1000500,18421,56662,2019-03-19,2025-12-31,2019-03-19,2025-12-31,True,18421,56662,True,True,False


Their candidate rows: 8; preview limit: 30


,membership_row,permno,permco,effective_start,effective_end,ccm_row,gvkey,lpermco,linkprim,linktype,linkdt,linkenddt,date_overlap,boundary_separation_days
67,52,13964,54433,2018-01-01,2025-12-31,444,018043,54433,J,LC,2013-07-01,E,True,NaN
76,59,14542,45483,2018-01-01,2025-12-31,712,160329,45483,J,LC,2014-04-03,E,True,NaN
99,75,15980,48779,2018-01-01,2022-06-17,730,165052,48779,J,LC,2016-04-08,E,True,NaN
586,472,90442,21287,2018-01-01,2019-03-19,390,012886,21287,P,LC,2004-11-12,2004-11-30,False,4780.0
587,472,90442,21287,2018-01-01,2019-03-19,391,012886,21287,J,LC,2004-12-01,E,True,NaN
615,500,92778,47011,2018-01-01,2022-04-08,728,164296,47011,J,LC,2008-09-18,2022-04-08,True,NaN
626,511,17676,56329,2018-05-01,2018-05-01,788,326688,56329,P,LC,2018-05-02,E,False,1.0
657,539,18421,56662,2019-03-19,2025-12-31,566,034636,56662,J,LC,2019-03-19,E,True,NaN


In [212]:
secondary_links = date_valid_links.loc[~date_valid_links["is_primary"]].copy()
one_day_boundaries = candidate_links.loc[
    candidate_links["boundary_separation_days"].eq(1), link_review_columns,
].copy()
print(f"Date-overlapping non-P/C candidate rows: {len(secondary_links)}")
with pd.option_context("display.max_columns", None):
    display(secondary_links[link_review_columns].head(30))
print(f"Candidate pairs with one-day endpoint separation: {len(one_day_boundaries)}")
with pd.option_context("display.max_columns", None):
    display(one_day_boundaries.head(30))

Date-overlapping non-P/C candidate rows: 6


,membership_row,permno,permco,effective_start,effective_end,ccm_row,gvkey,lpermco,linkprim,linktype,linkdt,linkenddt,date_overlap,boundary_separation_days
67,52,13964,54433,2018-01-01,2025-12-31,444,018043,54433,J,LC,2013-07-01,E,True,NaN
76,59,14542,45483,2018-01-01,2025-12-31,712,160329,45483,J,LC,2014-04-03,E,True,NaN
99,75,15980,48779,2018-01-01,2022-06-17,730,165052,48779,J,LC,2016-04-08,E,True,NaN
587,472,90442,21287,2018-01-01,2019-03-19,391,012886,21287,J,LC,2004-12-01,E,True,NaN
615,500,92778,47011,2018-01-01,2022-04-08,728,164296,47011,J,LC,2008-09-18,2022-04-08,True,NaN
657,539,18421,56662,2019-03-19,2025-12-31,566,034636,56662,J,LC,2019-03-19,E,True,NaN


Candidate pairs with one-day endpoint separation: 1


,membership_row,permno,permco,effective_start,effective_end,ccm_row,gvkey,lpermco,linkprim,linktype,linkdt,linkenddt,date_overlap,boundary_separation_days
626,511,17676,56329,2018-05-01,2018-05-01,788,326688,56329,P,LC,2018-05-02,E,False,1.0


### 3.6 Multiple and concurrent P/C links

Count P/C candidate rows and distinct GVKEY within each membership interval. Several
GVKEY may represent successive company identities rather than simultaneous conflicts.
Compare their actual overlapping segments to distinguish sequential links from
concurrent links. Retain same-GVKEY overlaps as well as different-GVKEY overlaps;
neither is automatically resolved. This comparison does not measure the union of
covered days or establish complete temporal coverage.

In [213]:
primary_link_counts = primary_links.groupby("membership_row").agg(
    primary_candidate_rows=("ccm_row", "size"),
    distinct_gvkey=("gvkey_key", "nunique"),
    any_single_link_spans_interval=("spans_full_interval", "any"),
)
multiple_primary_intervals = primary_link_counts.loc[
    primary_link_counts["primary_candidate_rows"].gt(1)
].copy()
multiple_primary_links = primary_links.loc[
    primary_links["membership_row"].isin(multiple_primary_intervals.index)
].copy()

print(f"Intervals with multiple P/C candidate rows: {len(multiple_primary_intervals)}")
print(f"Intervals with multiple P/C GVKEY: {int(primary_link_counts['distinct_gvkey'].gt(1).sum())}")
display(multiple_primary_intervals.head(30))
with pd.option_context("display.max_columns", None):
    display(multiple_primary_links[link_review_columns].head(30))

Intervals with multiple P/C candidate rows: 0
Intervals with multiple P/C GVKEY: 0


,primary_candidate_rows,distinct_gvkey,any_single_link_spans_interval
membership_row,,,


,membership_row,permno,permco,effective_start,effective_end,ccm_row,gvkey,lpermco,linkprim,linktype,linkdt,linkenddt,date_overlap,boundary_separation_days


In [214]:
primary_segments = multiple_primary_links[[
    "membership_row", "ccm_row", "gvkey_key", "overlap_start", "overlap_end",
]].copy()
primary_pairs = primary_segments.merge(
    primary_segments, on="membership_row", suffixes=("_a", "_b"),
)
concurrent_primary_links = primary_pairs.loc[
    primary_pairs["ccm_row_a"].lt(primary_pairs["ccm_row_b"])
    & primary_pairs["overlap_start_a"].le(primary_pairs["overlap_end_b"])
    & primary_pairs["overlap_start_b"].le(primary_pairs["overlap_end_a"])
].copy()
concurrent_primary_links["different_gvkey"] = (
    concurrent_primary_links["gvkey_key_a"] != concurrent_primary_links["gvkey_key_b"]
)
print(f"Concurrent P/C link pairs: {len(concurrent_primary_links)}")
print(f"Concurrent pairs with different GVKEY: {int(concurrent_primary_links['different_gvkey'].sum())}")
with pd.option_context("display.max_columns", None):
    display(concurrent_primary_links.head(30))

Concurrent P/C link pairs: 0
Concurrent pairs with different GVKEY: 0


,membership_row,ccm_row_a,gvkey_key_a,overlap_start_a,overlap_end_a,ccm_row_b,gvkey_key_b,overlap_start_b,overlap_end_b,different_gvkey


### 3.7 Source-record and company-identifier exceptions

Inspect CCM records with missing keys, unresolved/reversed dates, or exact duplicates.
Counts here refer to the CCM extract, not expanded membership-link pairs. Separately
compare nonmissing CRSP PERMCO and CCM LPERMCO on date-overlapping candidate pairs.
A disagreement is a review flag, not a reason to select an alternative GVKEY.
Missing values are reported separately from nonmissing disagreements.

In [215]:
ccm_history["missing_link_key"] = ccm_history[["permno_key", "gvkey_key"]].isna().any(axis=1)
ccm_exception_flags = [
    "missing_link_key", "missing_link_start", "missing_link_end",
    "invalid_link_start", "invalid_link_end", "reversed_link", "duplicate_ccm_record",
]
ccm_exceptions = ccm_history.loc[
    ccm_history[ccm_exception_flags].any(axis=1),
    ["ccm_row"] + list(ccm.columns) + ccm_exception_flags,
].copy()
display(ccm_history[ccm_exception_flags].sum().rename("ccm_records").to_frame())
print(f"CCM source records requiring review: {len(ccm_exceptions)}; preview limit: 30")
with pd.option_context("display.max_columns", None):
    display(ccm_exceptions.head(30))

permco_comparable = date_valid_links[["member_permco_key", "ccm_permco_key"]].notna().all(axis=1)
permco_disagrees = (
    date_valid_links["member_permco_key"].ne(date_valid_links["ccm_permco_key"]).fillna(False)
)
permco_conflicts = date_valid_links.loc[
    permco_comparable & permco_disagrees, link_review_columns,
].copy()
print(f"Date-valid candidate pairs missing either PERMCO: {int((~permco_comparable).sum())}")
print(f"Comparable pairs: {int(permco_comparable.sum())}; disagreements: {len(permco_conflicts)}")
with pd.option_context("display.max_columns", None):
    display(permco_conflicts.head(30))

,ccm_records
missing_link_key,0
missing_link_start,0
missing_link_end,0
invalid_link_start,0
invalid_link_end,0
reversed_link,0
duplicate_ccm_record,0


CCM source records requiring review: 0; preview limit: 30


,ccm_row,gvkey,linkprim,liid,linktype,lpermno,lpermco,linkdt,linkenddt,missing_link_key,missing_link_start,missing_link_end,invalid_link_start,invalid_link_end,reversed_link,duplicate_ccm_record


Date-valid candidate pairs missing either PERMCO: 0
Comparable pairs: 665; disagreements: 0


,membership_row,permno,permco,effective_start,effective_end,ccm_row,gvkey,lpermco,linkprim,linktype,linkdt,linkenddt,date_overlap,boundary_separation_days


### Findings

- All 666 membership intervals and 664 PERMNO have a CCM candidate with a GVKEY before date checks. Requiring date overlap reduces coverage to 665/666 intervals; requiring P/C links further reduces it to 659/666 intervals (98.95%) and 657/664 PERMNO (98.95%), representing 652 GVKEY. No LINKTYPE filter was applied.
- Seven intervals lack a date-overlapping P/C link: six have overlapping J links, while one has a P link separated from membership by one day. These cases remain unresolved; no secondary links were promoted and no dates were adjusted.
- A single P/C link spans the full effective membership interval for 658/666 intervals (98.80%). Thus, one of the 659 P/C-matched intervals has only partial coverage. An overlap-based match should not be interpreted as complete historical coverage.
- Across the 788 CCM records, there are no missing link keys, missing or invalid dates, reversed intervals, or exact duplicate records. The 592 open-ended `E` values were preserved, with the study end used only for date comparisons.
- No membership interval has multiple date-overlapping P/C candidates or concurrent P/C links. All 665 date-overlapping candidate pairs have nonmissing, matching PERMCO values. These checks support a relatively straightforward candidate mapping, while the seven unmatched intervals and the partially covered interval still require review.

## Part 4 - Compustat Mapping

Check Compustat availability for the date-overlapping P/C candidate GVKEY from Part 3.
Run Parts 0-3 first. This candidate set is provisional: the unresolved links and partial
coverage identified in Part 3 have not been corrected or approved as final mappings.

Keep company identification separate from annual observations. Do not join every
membership interval to every annual row. Preserve all formats, fiscal years, source
identifiers, and observations. Record previews show at most 30 rows; full diagnostic
DataFrames remain available locally. No representative company record is selected.

### 4.1 Prepare the candidate company set and annual data

Create one diagnostic row per candidate GVKEY by grouping the P/C links. Restrict
Compustat with `isin`, which retains each eligible source row once even when several
securities or membership intervals refer to the same GVKEY. The resulting
`compustat_sample` includes all downloaded annual records for those candidate firms;
it is not yet a finalized annual panel.

Trim whitespace in separate keys. For digit-only CIK values, remove leading zeros
for comparison only; retain the source CIK and flag unusual encodings. Parse dates
and EMP into new columns. No source values are overwritten.

In [216]:
candidate_companies = primary_links.groupby("gvkey_key").agg(
    linked_membership_intervals=("membership_row", "nunique"),
    linked_permno=("permno_key", "nunique"),
).reset_index()

compustat_for_mapping = compustat.copy()
compustat_for_mapping["compustat_row"] = range(1, len(compustat_for_mapping) + 1)
compustat_for_mapping["gvkey_key"] = (
    compustat_for_mapping["gvkey"].str.strip().replace("", pd.NA)
)
compustat_sample = compustat_for_mapping.loc[
    compustat_for_mapping["gvkey_key"].isin(candidate_companies["gvkey_key"])
].copy()

for column in ["conm", "cik", "cusip", "tic", "indfmt", "datafmt", "consol", "curcd", "fyear"]:
    compustat_sample[f"{column}_key"] = (
        compustat_sample[column].str.strip().replace("", pd.NA)
    )
cik_is_digits = compustat_sample["cik_key"].str.fullmatch(r"[0-9]+").fillna(False)
compustat_sample["cik_normalized"] = compustat_sample["cik_key"].copy()
compustat_sample.loc[cik_is_digits, "cik_normalized"] = (
    compustat_sample.loc[cik_is_digits, "cik_key"].str.lstrip("0").replace("", "0")
)
compustat_sample["nonstandard_cik"] = (
    compustat_sample["cik_key"].notna()
    & (~cik_is_digits | compustat_sample["cik_normalized"].eq("0").fillna(False))
)

In [217]:
report_date_text = compustat_sample["datadate"].str.strip().replace("", pd.NA)
compustat_sample["report_date"] = pd.to_datetime(
    report_date_text, format="%Y-%m-%d", errors="coerce",
)
compustat_sample["missing_report_date"] = report_date_text.isna()
compustat_sample["invalid_report_date"] = (
    report_date_text.notna() & compustat_sample["report_date"].isna()
)
compustat_sample["emp_text"] = compustat_sample["emp"].str.strip().replace("", pd.NA)
compustat_sample["emp_numeric"] = pd.to_numeric(compustat_sample["emp_text"], errors="coerce")
compustat_sample["emp_blank"] = compustat_sample["emp_text"].isna()
compustat_sample["emp_unparseable"] = (
    compustat_sample["emp_text"].notna() & compustat_sample["emp_numeric"].isna()
)

preparation_summary = pd.Series({
    "candidate_gvkey": len(candidate_companies),
    "downloaded_annual_rows_for_candidates": len(compustat_sample),
    "rows_missing_report_date": int(compustat_sample["missing_report_date"].sum()),
    "rows_with_unparseable_report_date": int(compustat_sample["invalid_report_date"].sum()),
    "rows_with_nonstandard_cik": int(compustat_sample["nonstandard_cik"].sum()),
}, name="count")
display(preparation_summary.to_frame())

,count
candidate_gvkey,652
downloaded_annual_rows_for_candidates,5725
rows_missing_report_date,0
rows_with_unparseable_report_date,0
rows_with_nonstandard_cik,0


### 4.2 Company coverage

Merge the candidate-company list with annual-row counts using a one-to-one join.
Coverage means at least one row in the downloaded Compustat file, not a complete
financial panel. Its denominator is the distinct candidate GVKEY from Part 3.
Missing firms may reflect extract coverage or historical identifiers; no cause is
assigned automatically. Membership context is retained for local follow-up.

In [218]:
compustat_company_counts = compustat_sample.groupby("gvkey_key").size().rename("annual_rows")
compustat_coverage = candidate_companies.merge(
    compustat_company_counts.reset_index(), on="gvkey_key", how="left", validate="one_to_one",
)
compustat_coverage["annual_rows"] = compustat_coverage["annual_rows"].fillna(0).astype("int64")
compustat_coverage["has_compustat"] = compustat_coverage["annual_rows"].gt(0)
unmatched_compustat = compustat_coverage.loc[~compustat_coverage["has_compustat"]].copy()

matched_company_count = int(compustat_coverage["has_compustat"].sum())
company_denominator = len(compustat_coverage)
display(pd.DataFrame([{
    "matched_gvkey": matched_company_count,
    "candidate_gvkey": company_denominator,
    "unmatched_gvkey": len(unmatched_compustat),
    "coverage_pct": 100 * matched_company_count / company_denominator if company_denominator else float("nan"),
}]).round({"coverage_pct": 2}))

unmatched_compustat_links = primary_links.loc[
    primary_links["gvkey_key"].isin(unmatched_compustat["gvkey_key"]),
    ["membership_row", "permno", "permco", "gvkey", "overlap_start", "overlap_end", "linktype"],
].copy()
print(f"Unmatched GVKEY: {len(unmatched_compustat)}; preview limit: 30")
display(unmatched_compustat.head(30))
print(f"Related membership-link rows: {len(unmatched_compustat_links)}; preview limit: 30")
display(unmatched_compustat_links.head(30))

,matched_gvkey,candidate_gvkey,unmatched_gvkey,coverage_pct
0,640,652,12,98.16


Unmatched GVKEY: 12; preview limit: 30


,gvkey_key,linked_membership_intervals,linked_permno,annual_rows,has_compustat
4,001177,1,1,0,False
211,009445,1,1,0,False
232,010466,1,1,0,False
350,024318,1,1,0,False
362,025056,1,1,0,False
369,025356,1,1,0,False
374,026304,1,1,0,False
402,029173,1,1,0,False
503,065967,1,1,0,False
547,140760,1,1,0,False


Related membership-link rows: 12; preview limit: 30


,membership_row,permno,permco,gvkey,overlap_start,overlap_end,linktype
98,74,15797,55470,026304,2018-01-01,2018-04-03,LC
181,118,23085,21570,009445,2018-01-01,2018-12-31,LU
252,169,37284,21732,010466,2018-01-01,2018-09-28,LC
420,318,77418,11412,025056,2018-01-01,2018-06-15,LC
426,324,77668,11569,025356,2018-01-01,2018-12-21,LC
453,346,79915,30044,029173,2018-01-01,2019-02-14,LU
509,400,85675,15867,065967,2018-01-01,2018-10-10,LC
521,411,86314,34859,024318,2018-01-01,2018-09-14,LC
547,435,88668,40148,140760,2018-01-01,2018-06-06,LC
549,437,88845,41072,001177,2018-01-01,2018-11-30,LC


### 4.3 Company identity consistency

Count nonmissing names and identifiers within each matched GVKEY across the downloaded
years and formats. Different names may reflect name changes; different CIK may reflect
historical identifiers or distinct entities. Counts alone do not determine which value
is correct. CUSIP and ticker may also change over time. The table summarizes available
identity information without choosing a preferred name, CIK, CUSIP, or annual row.

In [219]:
company_identity_summary = compustat_sample.groupby("gvkey_key").agg(
    annual_rows=("compustat_row", "size"),
    company_names=("conm_key", "nunique"),
    raw_cik_values=("cik_key", "nunique"),
    normalized_cik_values=("cik_normalized", "nunique"),
    cik_present_rows=("cik_key", "count"),
    cusip_values=("cusip_key", "nunique"),
    ticker_values=("tic_key", "nunique"),
    first_report_date=("report_date", "min"),
    last_report_date=("report_date", "max"),
)
company_identity_summary["cik_missing_rows"] = (
    company_identity_summary["annual_rows"] - company_identity_summary["cik_present_rows"]
)
identity_review = company_identity_summary.loc[
    company_identity_summary["company_names"].ne(1)
    | company_identity_summary["normalized_cik_values"].ne(1)
    | company_identity_summary["cik_missing_rows"].gt(0)
].copy()
display(pd.Series({
    "matched_gvkey": len(company_identity_summary),
    "gvkey_with_multiple_names": int(company_identity_summary["company_names"].gt(1).sum()),
    "gvkey_with_multiple_normalized_cik": int(company_identity_summary["normalized_cik_values"].gt(1).sum()),
    "gvkey_without_any_cik": int(company_identity_summary["normalized_cik_values"].eq(0).sum()),
    "gvkey_with_some_missing_cik": int(company_identity_summary["cik_missing_rows"].gt(0).sum()),
}, name="companies").to_frame())
print(f"Company identity summaries requiring review: {len(identity_review)}; preview limit: 30")
display(identity_review.head(30))

,companies
matched_gvkey,640
gvkey_with_multiple_names,0
gvkey_with_multiple_normalized_cik,0
gvkey_without_any_cik,1
gvkey_with_some_missing_cik,1


Company identity summaries requiring review: 1; preview limit: 30


,annual_rows,company_names,raw_cik_values,normalized_cik_values,cik_present_rows,cusip_values,ticker_values,first_report_date,last_report_date,cik_missing_rows
gvkey_key,,,,,,,,,,
028733,16,1,0,0,0,1,1,2018-12-31,2025-12-31,16


In [220]:
identity_records_for_review = compustat_sample.loc[
    compustat_sample["gvkey_key"].isin(identity_review.index)
    | compustat_sample["nonstandard_cik"],
    ["compustat_row", "gvkey", "datadate", "indfmt", "conm", "cik", "cik_normalized", "cusip", "tic"],
].sort_values(["gvkey", "datadate", "indfmt"])
print(f"Annual identity records for review: {len(identity_records_for_review)}; preview limit: 30")
with pd.option_context("display.max_columns", None):
    display(identity_records_for_review.head(30))

Annual identity records for review: 16; preview limit: 30


,compustat_row,gvkey,datadate,indfmt,conm,cik,cik_normalized,cusip,tic
30961,30962,028733,2018-12-31,FS,EQUITY RESIDENTIAL,<NA>,<NA>,29599Q934,EQR
30962,30963,028733,2018-12-31,INDL,EQUITY RESIDENTIAL,<NA>,<NA>,29599Q934,EQR
30963,30964,028733,2019-12-31,FS,EQUITY RESIDENTIAL,<NA>,<NA>,29599Q934,EQR
30964,30965,028733,2019-12-31,INDL,EQUITY RESIDENTIAL,<NA>,<NA>,29599Q934,EQR
30965,30966,028733,2020-12-31,FS,EQUITY RESIDENTIAL,<NA>,<NA>,29599Q934,EQR
30966,30967,028733,2020-12-31,INDL,EQUITY RESIDENTIAL,<NA>,<NA>,29599Q934,EQR
30967,30968,028733,2021-12-31,FS,EQUITY RESIDENTIAL,<NA>,<NA>,29599Q934,EQR
30968,30969,028733,2021-12-31,INDL,EQUITY RESIDENTIAL,<NA>,<NA>,29599Q934,EQR
30969,30970,028733,2022-12-31,FS,EQUITY RESIDENTIAL,<NA>,<NA>,29599Q934,EQR
30970,30971,028733,2022-12-31,INDL,EQUITY RESIDENTIAL,<NA>,<NA>,29599Q934,EQR


### 4.4 Annual observation structure

Check exact source-row duplicates separately from repeated company-year and company-date
keys. Exclude incomplete keys only from each duplicate diagnostic, and report their
counts. Repeated years or dates are not automatically errors. Keep all INDL/FS records.
Report fiscal-year counts as downloaded, including any fiscal years outside 2018-2025;
reporting dates and fiscal years are different concepts. A date-range summary does not
prove complete annual coverage or imply that every firm should have eight observations.

In [221]:
annual_key_checks = [
    ["gvkey_key", "fyear_key"],
    ["gvkey_key", "report_date"],
    ["gvkey_key", "report_date", "indfmt_key", "datafmt_key", "consol_key", "curcd_key"],
]
annual_duplicate_rows = []
for columns in annual_key_checks:
    keys = compustat_sample[columns]
    incomplete = keys.isna().any(axis=1)
    complete_keys = keys.loc[~incomplete]
    annual_duplicate_rows.append({
        "key": " + ".join(columns),
        "rows_with_incomplete_key": int(incomplete.sum()),
        "extra_rows_with_repeated_key": int(complete_keys.duplicated().sum()),
        "all_rows_with_repeated_key": int(complete_keys.duplicated(keep=False).sum()),
    })
display(pd.DataFrame(annual_duplicate_rows))
print("Exact duplicate source rows beyond the first:", int(
    compustat_sample.duplicated(subset=list(compustat.columns)).sum()
))
display(
    compustat_sample.groupby(["fyear_key", "indfmt_key"], dropna=False)
    .agg(annual_rows=("compustat_row", "size"), distinct_gvkey=("gvkey_key", "nunique"))
)

repeated_annual_rows = compustat_sample.loc[
    compustat_sample[["gvkey_key", "fyear_key"]].notna().all(axis=1)
    & compustat_sample.duplicated(["gvkey_key", "fyear_key"], keep=False),
    ["compustat_row", "gvkey", "fyear", "datadate", "indfmt", "datafmt", "consol", "curcd", "emp"],
].sort_values(["gvkey", "fyear", "datadate", "indfmt"])
print(f"Rows in repeated firm-year groups: {len(repeated_annual_rows)}; preview limit: 30")
display(repeated_annual_rows.head(30))

,key,rows_with_incomplete_key,extra_rows_with_repeated_key,all_rows_with_repeated_key
0,gvkey_key + fyear_key,0,878,1756
1,gvkey_key + report_date,0,878,1756
2,gvkey_key + report_date + indfmt_key + datafmt...,0,0,0


Exact duplicate source rows beyond the first: 0


annual_rows  distinct_gvkey
fyear_key indfmt_key                             
2017      INDL                 60              60
2018      FS                  109             109
          INDL                620             620
2019      FS                  107             107
          INDL                617             617
2020      FS                  111             111
          INDL                613             613
2021      FS                  112             112
          INDL                611             611
2022      FS                  112             112
          INDL                605             605
2023      FS                  110             110
          INDL                600             600
2024      FS                  109             109
          INDL                594             594
2025      FS                  108             108
          INDL                527             527

Rows in repeated firm-year groups: 1756; preview limit: 30


,compustat_row,gvkey,fyear,datadate,indfmt,datafmt,consol,curcd,emp
354,355,001447,2018,2018-12-31,FS,STD,C,USD,59.0000
355,356,001447,2018,2018-12-31,INDL,STD,C,USD,59.0000
356,357,001447,2019,2019-12-31,FS,STD,C,USD,64.5000
357,358,001447,2019,2019-12-31,INDL,STD,C,USD,64.5000
358,359,001447,2020,2020-12-31,FS,STD,C,USD,63.7000
359,360,001447,2020,2020-12-31,INDL,STD,C,USD,63.7000
360,361,001447,2021,2021-12-31,FS,STD,C,USD,64.0000
361,362,001447,2021,2021-12-31,INDL,STD,C,USD,64.0000
362,363,001447,2022,2022-12-31,FS,STD,C,USD,77.3000
363,364,001447,2022,2022-12-31,INDL,STD,C,USD,77.3000


### 4.5 INDL and FS comparisons

Within the candidate-company extract, compare formats at the same GVKEY, reporting
date, DATAFMT, and CONSOL. Currency is deliberately not a pairing key because currency
differences are part of the review. Pair only groups with exactly one INDL and one FS
row. Retain ambiguous, incomplete, and single-format groups for inspection; do not
create a many-to-many comparison or choose a preferred format.

Compare EMP numerically when parseable, retaining nonnumeric tokens for comparison.
For other fields use the prepared string keys; use normalized CIK. Two missing values
count as no observed difference, not as evidence that either value is known. One
missing value versus one observed value counts as a difference.

In [222]:
format_pair_keys = ["gvkey_key", "report_date", "datafmt_key", "consol_key"]
format_rows = compustat_sample.loc[compustat_sample["indfmt_key"].isin(["INDL", "FS"])].copy()
complete_format_keys = format_rows[format_pair_keys].notna().all(axis=1)
format_pair_counts = (
    format_rows.loc[complete_format_keys]
    .groupby(format_pair_keys + ["indfmt_key"]).size()
    .unstack("indfmt_key", fill_value=0)
    .reindex(columns=["INDL", "FS"], fill_value=0)
)
both_formats = format_pair_counts["INDL"].gt(0) & format_pair_counts["FS"].gt(0)
unique_format_pairs = format_pair_counts.loc[
    format_pair_counts["INDL"].eq(1) & format_pair_counts["FS"].eq(1)
].reset_index()
ambiguous_format_groups = format_pair_counts.loc[
    both_formats & (format_pair_counts["INDL"].gt(1) | format_pair_counts["FS"].gt(1))
].copy()
display(pd.Series({
    "INDL_FS_rows_with_incomplete_pair_keys": int((~complete_format_keys).sum()),
    "groups_with_both_formats": int(both_formats.sum()),
    "one_to_one_format_pairs": len(unique_format_pairs),
    "ambiguous_groups_with_both_formats": len(ambiguous_format_groups),
    "INDL_only_groups": int((format_pair_counts["INDL"].gt(0) & format_pair_counts["FS"].eq(0)).sum()),
    "FS_only_groups": int((format_pair_counts["FS"].gt(0) & format_pair_counts["INDL"].eq(0)).sum()),
}, name="count").to_frame())
display(ambiguous_format_groups.head(30))

,count
INDL_FS_rows_with_incomplete_pair_keys,0
groups_with_both_formats,878
one_to_one_format_pairs,878
ambiguous_groups_with_both_formats,0
INDL_only_groups,3969
FS_only_groups,0


,,,indfmt_key,INDL,FS
gvkey_key,report_date,datafmt_key,consol_key,,


In [223]:
paired_format_rows = format_rows.loc[complete_format_keys].merge(
    unique_format_pairs[format_pair_keys], on=format_pair_keys, how="inner", validate="many_to_one",
)
paired_format_rows["emp_comparison"] = (
    paired_format_rows["emp_numeric"].astype("string")
    .where(paired_format_rows["emp_numeric"].notna(), paired_format_rows["emp_text"])
)
comparison_columns = [
    "emp_comparison", "curcd_key", "fyear_key", "cik_normalized", "cusip_key", "conm_key",
]
format_columns = format_pair_keys + comparison_columns + ["compustat_row", "emp", "curcd"]
indl_rows = paired_format_rows.loc[paired_format_rows["indfmt_key"].eq("INDL"), format_columns]
fs_rows = paired_format_rows.loc[paired_format_rows["indfmt_key"].eq("FS"), format_columns]
indl_fs_comparison = indl_rows.merge(
    fs_rows, on=format_pair_keys, suffixes=("_indl", "_fs"), validate="one_to_one",
)
format_difference_flags = []
for column in comparison_columns:
    left = indl_fs_comparison[f"{column}_indl"]
    right = indl_fs_comparison[f"{column}_fs"]
    flag = f"different_{column}"
    indl_fs_comparison[flag] = ~(
        left.eq(right).fillna(False) | (left.isna() & right.isna())
    )
    format_difference_flags.append(flag)
format_differences = indl_fs_comparison.loc[
    indl_fs_comparison[format_difference_flags].any(axis=1)
].copy()
print(f"One-to-one pairs compared: {len(indl_fs_comparison)}")
print(f"Pairs differing in at least one selected field: {len(format_differences)}")
display(indl_fs_comparison[format_difference_flags].sum().rename("differing_pairs").to_frame())
with pd.option_context("display.max_columns", None):
    display(format_differences[
        format_pair_keys + ["emp_indl", "emp_fs", "curcd_indl", "curcd_fs"] + format_difference_flags
    ].head(30))

One-to-one pairs compared: 878
Pairs differing in at least one selected field: 6


,differing_pairs
different_emp_comparison,6
different_curcd_key,0
different_fyear_key,0
different_cik_normalized,0
different_cusip_key,0
different_conm_key,0


,gvkey_key,report_date,datafmt_key,consol_key,emp_indl,emp_fs,curcd_indl,curcd_fs,different_emp_comparison,different_curcd_key,different_fyear_key,different_cik_normalized,different_cusip_key,different_conm_key
36,002176,2022-12-31,STD,C,382.6510,383.0000,USD,USD,True,False,False,False,False,False
74,003243,2020-12-31,STD,C,210.1530,210.0000,USD,USD,True,False,False,False,False,False
685,143356,2022-12-31,STD,C,39.8540,39.5830,USD,USD,True,False,False,False,False,False
691,144009,2020-12-31,STD,C,46.1000,46.4000,USD,USD,True,False,False,False,False,False
695,144009,2024-12-31,STD,C,48.9000,49.0000,USD,USD,True,False,False,False,False,False
808,174729,2020-12-31,STD,C,15.5000,20.8000,USD,USD,True,False,False,False,False,False


### 4.6 EMP availability in the candidate sample

Report two descriptive scopes: (1) all downloaded annual rows for candidate GVKEY;
(2) rows whose reporting date falls within at least one P/C membership-link overlap
from Part 3. Mark the second scope with a Boolean flag rather than expanding annual
rows through an interval join. This reporting-date diagnostic is not an approved
annual-panel rule and does not establish full-fiscal-year membership.

Row-level EMP percentages include all formats, so repeated firm-years remain in the
denominator. Report missing/blank and nonblank unparseable EMP separately. Zero is
an observed numeric value, not missing; negative values are flagged for review. No
EMP values are imputed. Company counts cover firms with at least one row in the stated
scope; firms absent from Compustat are reported separately in Section 4.2.

In [224]:
compustat_sample["report_date_in_linked_membership"] = False
for gvkey, interval_start, interval_end in primary_links[
    ["gvkey_key", "overlap_start", "overlap_end"]
].itertuples(index=False, name=None):
    within_interval = (
        compustat_sample["gvkey_key"].eq(gvkey)
        & compustat_sample["report_date"].between(interval_start, interval_end)
    )
    compustat_sample.loc[within_interval, "report_date_in_linked_membership"] = True

emp_scopes = {
    "All downloaded rows for candidate companies": compustat_sample,
    "Reporting date within linked membership": compustat_sample.loc[
        compustat_sample["report_date_in_linked_membership"]
    ],
}
emp_summary_rows = []
for scope, annual_rows in emp_scopes.items():
    numeric_counts = annual_rows.groupby("gvkey_key")["emp_numeric"].count()
    emp_summary_rows.append({
        "scope": scope,
        "annual_rows": len(annual_rows),
        "gvkey_with_rows": annual_rows["gvkey_key"].nunique(),
        "blank_emp_rows": int(annual_rows["emp_blank"].sum()),
        "unparseable_emp_rows": int(annual_rows["emp_unparseable"].sum()),
        "numeric_unavailable_pct": 100 * annual_rows["emp_numeric"].isna().mean(),
        "zero_emp_rows": int(annual_rows["emp_numeric"].eq(0).sum()),
        "negative_emp_rows": int(annual_rows["emp_numeric"].lt(0).sum()),
        "gvkey_with_any_numeric_emp": int(numeric_counts.gt(0).sum()),
        "gvkey_without_numeric_emp": int(numeric_counts.eq(0).sum()),
    })
with pd.option_context("display.max_columns", None):
    display(pd.DataFrame(emp_summary_rows).round({"numeric_unavailable_pct": 2}))

,scope,annual_rows,gvkey_with_rows,blank_emp_rows,unparseable_emp_rows,numeric_unavailable_pct,zero_emp_rows,negative_emp_rows,gvkey_with_any_numeric_emp,gvkey_without_numeric_emp
0,All downloaded rows for candidate companies,5725,640,55,0,0.96,0,0,639,1
1,Reporting date within linked membership,4753,619,0,0,0.00,0,0,619,0


In [225]:
emp_by_year_format = compustat_sample.groupby(
    ["fyear_key", "indfmt_key"], dropna=False,
).agg(
    annual_rows=("compustat_row", "size"),
    gvkey_with_rows=("gvkey_key", "nunique"),
    numeric_emp_rows=("emp_numeric", "count"),
    blank_emp_rows=("emp_blank", "sum"),
    unparseable_emp_rows=("emp_unparseable", "sum"),
)
emp_by_year_format["numeric_unavailable_pct"] = 100 * (
    1 - emp_by_year_format["numeric_emp_rows"] / emp_by_year_format["annual_rows"]
)
display(emp_by_year_format.round({"numeric_unavailable_pct": 2}))

emp_exceptions = compustat_sample.loc[
    compustat_sample["emp_unparseable"] | compustat_sample["emp_numeric"].lt(0).fillna(False),
    ["compustat_row", "gvkey", "datadate", "fyear", "indfmt", "emp", "emp_unparseable"],
].copy()
print(f"Unparseable or negative EMP records: {len(emp_exceptions)}; preview limit: 30")
display(emp_exceptions.head(30))

annual_rows  gvkey_with_rows  numeric_emp_rows  \
fyear_key indfmt_key                                                   
2017      INDL                 60               60                60   
2018      FS                  109              109               108   
          INDL                620              620               610   
2019      FS                  107              107               106   
          INDL                617              617               607   
2020      FS                  111              111               110   
          INDL                613              613               605   
2021      FS                  112              112               112   
          INDL                611              611               600   
2022      FS                  112              112               112   
          INDL                605              605               597   
2023      FS                  110              110               110   
          INDL                600              600               596   
2024      FS                  109              109               109   
          INDL                594              594               593   
2025      FS                  108              108               108   
          INDL                527              527               527   

                      blank_emp_rows  unparseable_emp_rows  \
fyear_key indfmt_key                                         
2017      INDL                     0                     0   
2018      FS                       1                     0   
          INDL                    10                     0   
2019      FS                       1                     0   
          INDL                    10                     0   
2020      FS                       1                     0   
          INDL                     8                     0   
2021      FS                       0                     0   
          INDL                    11                     0   
2022      FS                       0                     0   
          INDL                     8                     0   
2023      FS                       0                     0   
          INDL                     4                     0   
2024      FS                       0                     0   
          INDL                     1                     0   
2025      FS                       0                     0   
          INDL                     0                     0   

                      numeric_unavailable_pct  
fyear_key indfmt_key                           
2017      INDL                            0.0  
2018      FS                             0.92  
          INDL                           1.61  
2019      FS                             0.93  
          INDL                           1.62  
2020      FS                              0.9  
          INDL                           1.31  
2021      FS                              0.0  
          INDL                            1.8  
2022      FS                              0.0  
          INDL                           1.32  
2023      FS                              0.0  
          INDL                           0.67  
2024      FS                              0.0  
          INDL                           0.17  
2025      FS                              0.0  
          INDL                            0.0

Unparseable or negative EMP records: 0; preview limit: 30


,compustat_row,gvkey,datadate,fyear,indfmt,emp,emp_unparseable


### Findings

- Compustat contains 5,725 annual records for 640 of the 652 candidate GVKEY (98.16%); 12 GVKEY are absent from this extract. Identifier coverage does not establish a complete annual panel.
- None of the 640 matched GVKEY has multiple observed company names or normalized CIK values. One GVKEY has no CIK. No report dates are missing or unparseable, and no nonstandard CIK encodings were flagged.
- There are no exact duplicate source rows. Repeated company-year and company-date keys each contribute 878 extra rows. All 878 INDL/FS pairs are one-to-one under the specified pairing keys; six pairs differ in EMP, with no differences in the other compared fields. Both formats remain retained pending a selection rule.
- EMP is blank in 55 of 5,725 candidate-company records (0.96%); one of the 640 firms has no numeric EMP. Among the 4,753 records whose reporting dates fall within a P/C membership-link interval, all have numeric EMP, but these records cover only 619 firms. This narrower denominator excludes firms without qualifying reporting dates and does not prove complete workforce coverage.
- The sample includes 60 records with fiscal year 2017. Fiscal-year selection, format selection, and the company-reference rule remain undecided; no annual records or conflicting values were removed.

## Part 5 - Revelio Mapping

Assess the ticker-filtered Revelio extract against all candidate GVKEY from Part 3,
including firms absent from Compustat. Run Parts 0-4 first. Match GVKEY directly, then
compare observed identity information. Additional CIK/CUSIP matches are separate review
candidates, not replacements for direct links. Never match on ticker alone.

No preferred Compustat reference row or Revelio RCID is selected. Preserve uncertain
records, original identifiers, and parent relationships. These data do not establish
historical identity or hierarchy validity throughout 2018-2025. No external queries,
exports, or downloads are performed. Local record previews are limited to 30 rows.

### 5.1 Prepare Revelio identifiers and Compustat evidence

Use separate string keys. CIK normalization removes leading zeros only from digit-only
values. Flag zero or nonnumeric CIK for review. CUSIP comparison uses the full uppercase
value; do not truncate to issuer prefixes. Only eight- or nine-character CUSIP strings
with standard identifier characters enter supplementary exact-match searches.

For each Compustat GVKEY, retain distinct observed names and identifiers as tuples.
Grouping repeated annual occurrences summarizes evidence; it does not choose a source
row or imply that an identifier was valid in every year. All annual records remain
in `compustat_sample` for dated inspection.

In [226]:
revelio_for_mapping = revelio.copy()
revelio_for_mapping["revelio_row"] = range(1, len(revelio_for_mapping) + 1)
for column in ["rcid", "gvkey", "cik", "cusip", "ticker", "company", "ultimate_parent_rcid"]:
    revelio_for_mapping[f"{column}_key"] = (
        revelio_for_mapping[column].str.strip().replace("", pd.NA)
    )
revelio_for_mapping["cusip_key"] = revelio_for_mapping["cusip_key"].str.upper()
revelio_cik_digits = revelio_for_mapping["cik_key"].str.fullmatch(r"[0-9]+").fillna(False)
revelio_for_mapping["cik_normalized"] = revelio_for_mapping["cik_key"].copy()
revelio_for_mapping.loc[revelio_cik_digits, "cik_normalized"] = (
    revelio_for_mapping.loc[revelio_cik_digits, "cik_key"].str.lstrip("0").replace("", "0")
)
revelio_for_mapping["valid_cik"] = (
    revelio_cik_digits & revelio_for_mapping["cik_normalized"].ne("0").fillna(False)
)
revelio_for_mapping["valid_cusip"] = (
    revelio_for_mapping["cusip_key"].str.fullmatch(r"[0-9A-Z*@#]{8,9}").fillna(False)
)
revelio_for_mapping["duplicate_source_row"] = revelio_for_mapping.duplicated(
    subset=list(revelio.columns), keep=False,
)
display(pd.Series({
    "mapping_rows": len(revelio_for_mapping),
    "unique_rcid": revelio_for_mapping["rcid_key"].nunique(),
    "rows_missing_rcid": int(revelio_for_mapping["rcid_key"].isna().sum()),
    "rows_missing_gvkey": int(revelio_for_mapping["gvkey_key"].isna().sum()),
    "nonmissing_nonstandard_cik": int((revelio_for_mapping["cik_key"].notna() & ~revelio_for_mapping["valid_cik"]).sum()),
    "nonmissing_nonstandard_cusip": int((revelio_for_mapping["cusip_key"].notna() & ~revelio_for_mapping["valid_cusip"]).sum()),
    "rows_in_exact_duplicate_groups": int(revelio_for_mapping["duplicate_source_row"].sum()),
}, name="count").to_frame())

,count
mapping_rows,1124
unique_rcid,1124
rows_missing_rcid,0
rows_missing_gvkey,214
nonmissing_nonstandard_cik,0
nonmissing_nonstandard_cusip,0
rows_in_exact_duplicate_groups,0


In [227]:
compustat_identity_evidence = compustat_sample[[
    "gvkey_key", "conm_key", "cik_key", "cik_normalized", "nonstandard_cik", "cusip_key", "tic_key",
]].copy()
compustat_identity_evidence["cusip_key"] = compustat_identity_evidence["cusip_key"].str.upper()
compustat_identity_evidence["cik_for_comparison"] = (
    compustat_identity_evidence["cik_normalized"].where(~compustat_identity_evidence["nonstandard_cik"])
)
compustat_identity_evidence["cusip_for_comparison"] = (
    compustat_identity_evidence["cusip_key"].where(
        compustat_identity_evidence["cusip_key"].str.fullmatch(r"[0-9A-Z*@#]{8,9}").fillna(False)
    )
)

identity_sets = compustat_identity_evidence.groupby("gvkey_key").agg(
    observed_names=("conm_key", lambda values: tuple(sorted(values.dropna().unique()))),
    observed_raw_cik=("cik_key", lambda values: tuple(sorted(values.dropna().unique()))),
    observed_cik=("cik_for_comparison", lambda values: tuple(sorted(values.dropna().unique()))),
    observed_cusip=("cusip_for_comparison", lambda values: tuple(sorted(values.dropna().unique()))),
    observed_tickers=("tic_key", lambda values: tuple(sorted(values.dropna().unique()))),
).reset_index()
company_evidence = candidate_companies[["gvkey_key"]].merge(
    identity_sets, on="gvkey_key", how="left", validate="one_to_one", indicator="compustat_evidence",
)
for column in ["observed_names", "observed_raw_cik", "observed_cik", "observed_cusip", "observed_tickers"]:
    company_evidence[column] = [value if isinstance(value, tuple) else () for value in company_evidence[column]]

### 5.2 Direct GVKEY candidate coverage

Join the unique candidate-company list to the Revelio extract. A firm counts as covered
only if at least one direct-GVKEY row has a nonmissing RCID. Use all candidate GVKEY as
the denominator, not just firms found in Compustat. Retain multiple RCID candidates and
missing-RCID mapping records. This is coverage of the downloaded extract, not of the
complete Revelio database, and it does not certify identity.

In [228]:
revelio_candidates = company_evidence.merge(
    revelio_for_mapping.loc[revelio_for_mapping["gvkey_key"].notna()],
    on="gvkey_key", how="left", validate="one_to_many", indicator="revelio_candidate_status",
)
direct_matches = revelio_candidates.loc[
    revelio_candidates["revelio_candidate_status"].eq("both")
].copy()
direct_rcid_counts = direct_matches.groupby("gvkey_key")["rcid_key"].nunique()
revelio_coverage = candidate_companies[["gvkey_key"]].copy()
revelio_coverage["direct_rcid_candidates"] = (
    revelio_coverage["gvkey_key"].map(direct_rcid_counts).fillna(0).astype("int64")
)
revelio_coverage["has_direct_rcid"] = revelio_coverage["direct_rcid_candidates"].gt(0)
unmatched_revelio = revelio_coverage.loc[~revelio_coverage["has_direct_rcid"]].copy()
direct_matched_gvkey = revelio_coverage.loc[revelio_coverage["has_direct_rcid"], "gvkey_key"]
revelio_denominator = len(revelio_coverage)
display(pd.DataFrame([{
    "matched_gvkey": len(direct_matched_gvkey),
    "candidate_gvkey": revelio_denominator,
    "coverage_pct": 100 * len(direct_matched_gvkey) / revelio_denominator if revelio_denominator else float("nan"),
    "unmatched_gvkey": len(unmatched_revelio),
    "unique_candidate_rcid": direct_matches["rcid_key"].nunique(),
    "gvkey_with_multiple_rcid": int(revelio_coverage["direct_rcid_candidates"].gt(1).sum()),
}]).round({"coverage_pct": 2}))

,matched_gvkey,candidate_gvkey,coverage_pct,unmatched_gvkey,unique_candidate_rcid,gvkey_with_multiple_rcid
0,539,652,82.67,113,539,0


### 5.3 Cross-check direct candidates against observed identities

Compare each direct-GVKEY mapping row with all observed Compustat identifiers for that
GVKEY, rather than repeatedly comparing annual rows. Distinguish missing evidence,
nonstandard identifiers, exact agreement, agreement after CIK zero normalization,
and disagreement with all observed values. A disagreement is an unresolved conflict
in this extract, not proof that either source is wrong or that parent/subsidiary identity
explains it. Multiple observed CIK are flagged separately.

Name comparison only ignores case and surrounding whitespace. Name differences are
review cues, not fuzzy matches or automatic rejections. CUSIP disagreements may reflect
different securities, lengths, or dates; no prefix-based equivalence is assumed.

In [229]:
cik_statuses = []
cusip_statuses = []
name_agreements = []
for row in direct_matches.itertuples(index=False):
    if row.compustat_evidence != "both":
        cik_status = "compustat_unavailable"
    elif pd.isna(row.cik_key):
        cik_status = "revelio_cik_missing"
    elif not row.valid_cik:
        cik_status = "revelio_cik_nonstandard"
    elif not row.observed_cik:
        cik_status = "compustat_cik_missing_or_nonstandard"
    elif row.cik_normalized not in row.observed_cik:
        cik_status = "conflicts_with_observed_cik"
    elif row.cik_key in row.observed_raw_cik:
        cik_status = "same_observed_cik"
    else:
        cik_status = "same_after_zero_normalization"
    cik_statuses.append(cik_status)

    if row.compustat_evidence != "both":
        cusip_status = "compustat_unavailable"
    elif pd.isna(row.cusip_key):
        cusip_status = "revelio_cusip_missing"
    elif not row.valid_cusip:
        cusip_status = "revelio_cusip_nonstandard"
    elif not row.observed_cusip:
        cusip_status = "compustat_cusip_missing_or_nonstandard"
    elif row.cusip_key in row.observed_cusip:
        cusip_status = "same_observed_full_cusip"
    else:
        cusip_status = "different_observed_cusip_review"
    cusip_statuses.append(cusip_status)
    name_agreements.append(
        pd.NA if pd.isna(row.company_key) or not row.observed_names
        else row.company_key.casefold() in [name.casefold() for name in row.observed_names]
    )

direct_matches["cik_status"] = cik_statuses
direct_matches["cusip_status"] = cusip_statuses
direct_matches["name_agrees"] = pd.array(name_agreements, dtype="boolean")
direct_matches["multiple_observed_cik"] = direct_matches["observed_cik"].map(len).gt(1)

In [230]:
for column in ["cik_status", "cusip_status"]:
    display(direct_matches.groupby(column).agg(
        mapping_rows=("revelio_row", "size"),
        distinct_gvkey=("gvkey_key", "nunique"),
        distinct_rcid=("rcid_key", "nunique"),
    ))

comparable_cik_statuses = [
    "same_observed_cik", "same_after_zero_normalization", "conflicts_with_observed_cik",
]
cik_comparable = direct_matches["cik_status"].isin(comparable_cik_statuses)
cik_conflicts = direct_matches.loc[
    direct_matches["cik_status"].eq("conflicts_with_observed_cik")
].copy()
display(pd.DataFrame([{
    "direct_mapping_rows": len(direct_matches),
    "rows_with_comparable_cik": int(cik_comparable.sum()),
    "cik_conflict_rows": len(cik_conflicts),
    "gvkey_with_cik_conflict": cik_conflicts["gvkey_key"].nunique(),
    "conflict_pct_of_comparable_rows": 100 * len(cik_conflicts) / cik_comparable.sum() if cik_comparable.any() else float("nan"),
    "rows_with_multiple_observed_compustat_cik": int(direct_matches["multiple_observed_cik"].sum()),
}]).round({"conflict_pct_of_comparable_rows": 2}))
identity_review_columns = [
    "gvkey_key", "rcid", "company", "observed_names", "cik", "observed_cik",
    "cik_status", "cusip", "observed_cusip", "cusip_status", "name_agrees",
]
print(f"CIK conflict rows: {len(cik_conflicts)}; preview limit: 30")
with pd.option_context("display.max_columns", None, "display.max_colwidth", 80):
    display(cik_conflicts[identity_review_columns].head(30))

,mapping_rows,distinct_gvkey,distinct_rcid
cik_status,,,
conflicts_with_observed_cik,28,28,28
same_observed_cik,511,511,511


,mapping_rows,distinct_gvkey,distinct_rcid
cusip_status,,,
different_observed_cusip_review,2,2,2
same_observed_full_cusip,537,537,537


,direct_mapping_rows,rows_with_comparable_cik,cik_conflict_rows,gvkey_with_cik_conflict,conflict_pct_of_comparable_rows,rows_with_multiple_observed_compustat_cik
0,539,539,28,28,5.19,0


CIK conflict rows: 28; preview limit: 30


,gvkey_key,rcid,company,observed_names,cik,observed_cik,cik_status,cusip,observed_cusip,cusip_status,name_agrees
14,001598,73989,"AMETEK, Inc.","(AMETEK INC,)",0000006082,"(1037868,)",conflicts_with_observed_cik,031100100,"(031100100,)",same_observed_full_cusip,False
70,003980,745033,The Walt Disney Co.,"(DISNEY (WALT) CO,)",0001001039,"(1744489,)",conflicts_with_observed_cik,254687106,"(254687106,)",same_observed_full_cusip,False
76,004093,433379,Duke Energy Corp.,"(DUKE ENERGY CORP,)",0000081020,"(1326160,)",conflicts_with_observed_cik,26441C204,"(26441C204,)",same_observed_full_cusip,False
89,004598,806872,FedEx Corp.,"(FEDEX CORP,)",0000230211,"(1048911,)",conflicts_with_observed_cik,31428X106,"(31428X106,)",same_observed_full_cusip,False
93,004674,1495498,Regions Financial Corp.,"(REGIONS FINANCIAL CORP,)",0000036032,"(1281761,)",conflicts_with_observed_cik,7591EP100,"(7591EP100,)",same_observed_full_cusip,False
107,005256,1496843,"W.W. Grainger, Inc.","(GRAINGER (W W) INC,)",0000276172,"(277135,)",conflicts_with_observed_cik,384802104,"(384802104,)",same_observed_full_cusip,False
125,005959,793427,Trane Technologies Plc,"(TRANE TECHNOLOGIES PLC,)",0001160497,"(1466258,)",conflicts_with_observed_cik,G8994E103,"(G8994E103,)",same_observed_full_cusip,True
136,006310,865831,"Kinder Morgan, Inc.","(KINDER MORGAN INC,)",0000054502,"(1506307,)",conflicts_with_observed_cik,49456B101,"(49456B101,)",same_observed_full_cusip,False
144,006669,642378,Lennar Corp.,"(LENNAR CORP,)",0000058696,"(920760,)",conflicts_with_observed_cik,526057104,"(526057104,)",same_observed_full_cusip,False
179,007974,199524,"NiSource, Inc.","(NISOURCE INC,)",0000823392,"(1111711,)",conflicts_with_observed_cik,65473P105,"(65473P105,)",same_observed_full_cusip,False


### 5.4 Multiple RCID and parent/subsidiary review

Check how many RCID candidates each GVKEY has, and whether the same RCID appears under
multiple sample GVKEY. Preserve the extract's ultimate-parent and child fields. A
reported parent different from RCID is a hierarchy review flag, not an instruction
to replace the company RCID with its parent. Child fields are retained as source text;
no child relationships are inferred by parsing them. Historical validity is unresolved.

In [231]:
multi_rcid_gvkey = revelio_coverage.loc[
    revelio_coverage["direct_rcid_candidates"].gt(1), "gvkey_key",
]
rcid_gvkey_counts = direct_matches.loc[direct_matches["rcid_key"].notna()].groupby("rcid_key")["gvkey_key"].nunique()
shared_rcids = rcid_gvkey_counts.loc[rcid_gvkey_counts.gt(1)].index
direct_matches["multiple_rcid_for_gvkey"] = direct_matches["gvkey_key"].isin(multi_rcid_gvkey)
direct_matches["rcid_shared_across_gvkey"] = direct_matches["rcid_key"].isin(shared_rcids)
direct_matches["parent_missing"] = direct_matches["ultimate_parent_rcid_key"].isna()
direct_matches["different_ultimate_parent"] = (
    direct_matches["rcid_key"].notna()
    & direct_matches["ultimate_parent_rcid_key"].notna()
    & direct_matches["rcid_key"].ne(direct_matches["ultimate_parent_rcid_key"]).fillna(False)
)
display(pd.Series({
    "gvkey_with_multiple_direct_rcid": len(multi_rcid_gvkey),
    "rcid_shared_across_sample_gvkey": len(shared_rcids),
    "direct_rows_with_missing_parent": int(direct_matches["parent_missing"].sum()),
    "direct_rows_with_different_parent": int(direct_matches["different_ultimate_parent"].sum()),
}, name="count").to_frame())
hierarchy_review = direct_matches.loc[
    direct_matches["multiple_rcid_for_gvkey"] | direct_matches["rcid_shared_across_gvkey"]
    | direct_matches["parent_missing"] | direct_matches["different_ultimate_parent"],
    ["gvkey_key", "rcid", "company", "ultimate_parent_rcid", "ultimate_parent_company_name",
     "child_rcid", "child_company", "multiple_rcid_for_gvkey", "rcid_shared_across_gvkey"],
].copy()
print(f"Hierarchy or RCID-ambiguity review rows: {len(hierarchy_review)}; preview limit: 30")
with pd.option_context("display.max_columns", None, "display.max_colwidth", 80):
    display(hierarchy_review.head(30))

,count
gvkey_with_multiple_direct_rcid,0
rcid_shared_across_sample_gvkey,0
direct_rows_with_missing_parent,0
direct_rows_with_different_parent,4


Hierarchy or RCID-ambiguity review rows: 4; preview limit: 30


,gvkey_key,rcid,company,ultimate_parent_rcid,ultimate_parent_company_name,child_rcid,child_company,multiple_rcid_for_gvkey,rcid_shared_across_gvkey
274,012151,1340225,"Coty, Inc.",22193191,Agnaten SE,565278,Younique LLC,False,False
322,017874,22226748,"T-Mobile US, Inc.",800066,Deutsche Telekom AG,906962,Sprint Communications LLC,False,False
530,125240,94777116,"TKO Group Holdings, Inc.",21037150,"Endeavor Group Holdings, Inc.",223939,World Wrestling Entertainment LLC,False,False
585,161844,448784,Las Vegas Sands Corp.,22266300,Estate of Adelson Sheldon Gary,782226,Marina Bay Sands Pte Ltd.,False,False


### 5.5 Supplementary CIK and full-CUSIP candidate searches

Search only the existing local Revelio extract. Collapse repeated annual occurrences
of each GVKEY-identifier pair by counting them, not by selecting a company record.
Retain all resulting RCID candidates. Missing/nonstandard identifiers never participate
in these joins. A common identifier may produce multiple candidates or connect different
GVKEY; report that conflict rather than accepting an alternative match.

These searches use Compustat evidence, so firms missing from Compustat cannot benefit
from them here. Exact full-CUSIP matching does not equate eight- and nine-character
values. No ticker-only or fuzzy-name matching is performed.

In [232]:
compustat_cik_keys = (
    compustat_identity_evidence.loc[compustat_identity_evidence["cik_for_comparison"].notna()]
    .groupby(["gvkey_key", "cik_for_comparison"]).size().rename("annual_evidence_rows").reset_index()
)
compustat_cusip_keys = (
    compustat_identity_evidence.loc[compustat_identity_evidence["cusip_for_comparison"].notna()]
    .groupby(["gvkey_key", "cusip_for_comparison"]).size().rename("annual_evidence_rows").reset_index()
)
lookup_columns = ["revelio_row", "rcid_key", "company", "gvkey_key", "cik_normalized", "cusip_key"]
revelio_lookup = revelio_for_mapping[lookup_columns].rename(columns={"gvkey_key": "revelio_gvkey_key"})
cik_candidates = compustat_cik_keys.merge(
    revelio_lookup.loc[revelio_for_mapping["valid_cik"] & revelio_for_mapping["rcid_key"].notna()],
    left_on="cik_for_comparison", right_on="cik_normalized", how="inner",
)
cusip_candidates = compustat_cusip_keys.merge(
    revelio_lookup.loc[revelio_for_mapping["valid_cusip"] & revelio_for_mapping["rcid_key"].notna()],
    left_on="cusip_for_comparison", right_on="cusip_key", how="inner",
)
for candidates in [cik_candidates, cusip_candidates]:
    candidates["different_nonmissing_revelio_gvkey"] = (
        candidates["revelio_gvkey_key"].notna()
        & candidates["gvkey_key"].ne(candidates["revelio_gvkey_key"]).fillna(False)
    )
    candidates["no_direct_gvkey_candidate"] = ~candidates["gvkey_key"].isin(direct_matched_gvkey)

for label, candidates in [("CIK", cik_candidates), ("Full CUSIP", cusip_candidates)]:
    print(f"{label}: {len(candidates)} identifier-evidence candidate rows")
    print("Rows with a different nonmissing Revelio GVKEY:", int(candidates["different_nonmissing_revelio_gvkey"].sum()))
    print("GVKEY without direct matches but with these candidates:", candidates.loc[
        candidates["no_direct_gvkey_candidate"], "gvkey_key",
    ].nunique())
    with pd.option_context("display.max_columns", None, "display.max_colwidth", 80):
        display(candidates.loc[
            candidates["no_direct_gvkey_candidate"] | candidates["different_nonmissing_revelio_gvkey"]
        ].head(30))

CIK: 536 identifier-evidence candidate rows
Rows with a different nonmissing Revelio GVKEY: 0
GVKEY without direct matches but with these candidates: 25


,gvkey_key,cik_for_comparison,annual_evidence_rows,revelio_row,rcid_key,company,revelio_gvkey_key,cik_normalized,cusip_key,different_nonmissing_revelio_gvkey,no_direct_gvkey_candidate
75,004503,2115436,8,1114,106152626,ExxonMobil Holdings Corp.,<NA>,2115436,30233Q108,False,True
96,005543,766704,16,1095,94934825,"Welltower, Inc.",<NA>,766704,95040Q104,False,True
203,010795,100517,8,1037,22214941,"United Airlines Holdings, Inc.",<NA>,100517,910047109,False,True
224,011687,109380,16,1123,22239946,Zions Bancorporation NA,<NA>,109380,989701107,False,True
237,013125,765880,16,349,96122418,"Healthpeak Properties, Inc.",<NA>,765880,42250P103,False,True
242,013714,2041610,8,848,102240445,Paramount Skydance Corp.,<NA>,2041610,69932A204,False,True
250,014489,1571996,8,332,21023141,"Dell Technologies, Inc.",<NA>,1571996,24703L202,False,True
254,014960,920148,8,619,96737402,"Labcorp Holdings, Inc.",<NA>,920148,504922105,False,True
299,025313,1757898,8,952,22254232,STERIS Plc (Ireland),<NA>,1757898,G8473T100,False,True
306,027786,895126,8,420,98625118,Expand Energy Corp.,<NA>,895126,165167735,False,True


Full CUSIP: 563 identifier-evidence candidate rows
Rows with a different nonmissing Revelio GVKEY: 0
GVKEY without direct matches but with these candidates: 26


,gvkey_key,cusip_for_comparison,annual_evidence_rows,revelio_row,rcid_key,company,revelio_gvkey_key,cik_normalized,cusip_key,different_nonmissing_revelio_gvkey,no_direct_gvkey_candidate
78,004503,30233Q108,8,1114,106152626,ExxonMobil Holdings Corp.,<NA>,2115436,30233Q108,False,True
102,005543,95040Q104,16,1095,94934825,"Welltower, Inc.",<NA>,766704,95040Q104,False,True
216,010795,910047109,8,1037,22214941,"United Airlines Holdings, Inc.",<NA>,100517,910047109,False,True
238,011687,989701107,16,1123,22239946,Zions Bancorporation NA,<NA>,109380,989701107,False,True
252,013125,42250P103,16,349,96122418,"Healthpeak Properties, Inc.",<NA>,765880,42250P103,False,True
257,013714,69932A204,8,848,102240445,Paramount Skydance Corp.,<NA>,2041610,69932A204,False,True
265,014489,24703L202,8,332,21023141,"Dell Technologies, Inc.",<NA>,1571996,24703L202,False,True
270,014960,504922105,8,619,96737402,"Labcorp Holdings, Inc.",<NA>,920148,504922105,False,True
318,025313,G8473T100,8,952,22254232,STERIS Plc (Ireland),<NA>,1757898,G8473T100,False,True
325,027786,165167735,8,420,98625118,Expand Energy Corp.,<NA>,895126,165167735,False,True


### 5.6 Candidate coverage by method and follow-up list

Use the same full candidate-GVKEY denominator for every method. The union is an upper
bound on local candidate availability, not an accepted match rate: it includes conflicts
and ambiguous alternatives. Show the additional firms beyond direct GVKEY matching
without promoting them. The 85% follow-up threshold comes from the project handoff;
falling below it prompts review of the ticker-filtered extraction, not an automatic
conclusion about the full Revelio database.

In [233]:
method_gvkey = {
    "Direct GVKEY": set(direct_matched_gvkey),
    "Normalized CIK candidates": set(cik_candidates["gvkey_key"]),
    "Exact full-CUSIP candidates": set(cusip_candidates["gvkey_key"]),
}
method_gvkey["Any local candidate method"] = set().union(*method_gvkey.values())
method_coverage_rows = []
for method, firms in method_gvkey.items():
    method_coverage_rows.append({
        "method": method,
        "gvkey_with_candidates": len(firms),
        "total_candidate_gvkey": revelio_denominator,
        "candidate_coverage_pct": 100 * len(firms) / revelio_denominator if revelio_denominator else float("nan"),
        "additional_gvkey_beyond_direct": len(firms - method_gvkey["Direct GVKEY"]),
    })
display(pd.DataFrame(method_coverage_rows).round({"candidate_coverage_pct": 2}))

follow_up_firms = company_evidence.loc[
    company_evidence["gvkey_key"].isin(unmatched_revelio["gvkey_key"])
].copy()
for label, candidates in [("cik", cik_candidates), ("cusip", cusip_candidates)]:
    counts = candidates.groupby("gvkey_key")["rcid_key"].nunique()
    follow_up_firms[f"{label}_candidate_rcids"] = follow_up_firms["gvkey_key"].map(counts).fillna(0).astype("int64")
follow_up_firms["has_local_alternative"] = (
    follow_up_firms["cik_candidate_rcids"].gt(0) | follow_up_firms["cusip_candidate_rcids"].gt(0)
)
print(f"GVKEY without direct RCID candidates: {len(follow_up_firms)}; preview limit: 30")
with pd.option_context("display.max_columns", None, "display.max_colwidth", 80):
    display(follow_up_firms.head(30))
if revelio_denominator and len(direct_matched_gvkey) / revelio_denominator < 0.85:
    print("Direct extract coverage is below 85%. Review missing firms before proposing a targeted WRDS query.")

,method,gvkey_with_candidates,total_candidate_gvkey,candidate_coverage_pct,additional_gvkey_beyond_direct
0,Direct GVKEY,539,652,82.67,0
1,Normalized CIK candidates,536,652,82.21,25
2,Exact full-CUSIP candidates,563,652,86.35,26
3,Any local candidate method,565,652,86.66,26


GVKEY without direct RCID candidates: 113; preview limit: 30


,gvkey_key,observed_names,observed_raw_cik,observed_cik,observed_cusip,observed_tickers,compustat_evidence,cik_candidate_rcids,cusip_candidate_rcids,has_local_alternative
4,001177,(),(),(),(),(),left_only,0,0,False
9,001380,"(HESS CORP,)","(0000004447,)","(4447,)","(42809H107,)","(HES,)",both,0,0,False
25,002019,"(BANK OF NEW YORK MELLON CORP,)","(0001390777,)","(1390777,)","(064058100,)","(BNY,)",both,0,0,False
29,002154,"(BEMIS CO INC,)","(0000011199,)","(11199,)","(081437105,)","(BMS,)",both,0,0,False
30,002176,"(BERKSHIRE HATHAWAY,)","(0001067983,)","(1067983,)","(084670702,)","(BRK.B,)",both,0,0,False
37,002435,"(BROWN FORMAN CORP,)","(0000014693,)","(14693,)","(115637209,)","(BF.B,)",both,0,0,False
54,003231,"(COMERICA INC,)","(0000028412,)","(28412,)","(200340107,)","(CMA,)",both,0,0,False
56,003310,"(CA INC,)","(0000356028,)","(356028,)","(12673P105,)","(CA.1,)",both,0,0,False
86,004503,"(EXXONMOBIL HOLDINGS CORP,)","(0002115436,)","(2115436,)","(30233Q108,)","(XOM,)",both,1,1,True
100,004990,"(GAP INC,)","(0000039911,)","(39911,)","(364760108,)","(GAP,)",both,0,0,False


Direct extract coverage is below 85%. Review missing firms before proposing a targeted WRDS query.


### 5.7 Consolidated direct-candidate review

Keep all direct candidates, with review flags for missing RCID, uncertain or conflicting
identifiers, multiple observed CIK, differing names, multiple RCID, and hierarchy issues.
Agreement on one identifier does not override disagreement on another. The review table
is not a rejected-record table, and unflagged candidates are not certified historical
matches. Supplemental identifier conflicts remain in the separate candidate tables.

In [234]:
direct_matches["missing_rcid"] = direct_matches["rcid_key"].isna()
direct_matches["cik_needs_review"] = ~direct_matches["cik_status"].isin([
    "same_observed_cik", "same_after_zero_normalization",
])
direct_matches["cusip_needs_review"] = ~direct_matches["cusip_status"].eq("same_observed_full_cusip")
direct_matches["name_needs_review"] = ~direct_matches["name_agrees"].fillna(False)
revelio_review_flags = [
    "missing_rcid", "cik_needs_review", "cusip_needs_review", "multiple_observed_cik",
    "name_needs_review", "multiple_rcid_for_gvkey", "rcid_shared_across_gvkey",
    "parent_missing", "different_ultimate_parent", "duplicate_source_row",
]
display(direct_matches[revelio_review_flags].sum().rename("direct_mapping_rows").to_frame())
revelio_exceptions = direct_matches.loc[
    direct_matches[revelio_review_flags].any(axis=1)
].copy()
print(f"Direct-candidate rows with review flags: {len(revelio_exceptions)}; preview limit: 30")
with pd.option_context("display.max_columns", None, "display.max_colwidth", 80):
    display(revelio_exceptions[identity_review_columns + [
        "ultimate_parent_rcid", "multiple_rcid_for_gvkey", "rcid_shared_across_gvkey",
    ]].head(30))

,direct_mapping_rows
missing_rcid,0
cik_needs_review,28
cusip_needs_review,2
multiple_observed_cik,0
name_needs_review,509
multiple_rcid_for_gvkey,0
rcid_shared_across_gvkey,0
parent_missing,0
different_ultimate_parent,4
duplicate_source_row,0


Direct-candidate rows with review flags: 512; preview limit: 30


,gvkey_key,rcid,company,observed_names,cik,observed_cik,cik_status,cusip,observed_cusip,cusip_status,name_agrees,ultimate_parent_rcid,multiple_rcid_for_gvkey,rcid_shared_across_gvkey
0,001045,21007446,"American Airlines Group, Inc.","(AMERICAN AIRLINES GROUP INC,)",0000006201,"(6201,)",same_observed_cik,02376R102,"(02376R102,)",same_observed_full_cusip,False,21007446,False,False
1,001075,368391,Pinnacle West Capital Corp.,"(PINNACLE WEST CAPITAL CORP,)",0000764622,"(764622,)",same_observed_cik,723484101,"(723484101,)",same_observed_full_cusip,False,368391,False,False
3,001161,1292281,"Advanced Micro Devices, Inc.","(ADVANCED MICRO DEVICES,)",0000002488,"(2488,)",same_observed_cik,007903107,"(007903107,)",same_observed_full_cusip,False,1292281,False,False
5,001209,1511315,"Air Products & Chemicals, Inc.","(AIR PRODUCTS & CHEMICALS INC,)",0000002969,"(2969,)",same_observed_cik,009158106,"(009158106,)",same_observed_full_cusip,False,1511315,False,False
6,001230,22142001,"Alaska Air Group, Inc.","(ALASKA AIR GROUP INC,)",0000766421,"(766421,)",same_observed_cik,011659109,"(011659109,)",same_observed_full_cusip,False,22142001,False,False
7,001300,20924380,"Honeywell International, Inc.","(HONEYWELL INTERNATIONAL INC,)",0000773840,"(773840,)",same_observed_cik,438516205,"(438516205,)",same_observed_full_cusip,False,20924380,False,False
8,001327,47589,"Skyworks Solutions, Inc.","(SKYWORKS SOLUTIONS INC,)",0000004127,"(4127,)",same_observed_cik,83088M102,"(83088M102,)",same_observed_full_cusip,False,47589,False,False
10,001440,479560,"American Electric Power Co., Inc.","(AMERICAN ELECTRIC POWER CO,)",0000004904,"(4904,)",same_observed_cik,025537101,"(025537101,)",same_observed_full_cusip,False,479560,False,False
11,001447,1104097,American Express Co.,"(AMERICAN EXPRESS CO,)",0000004962,"(4962,)",same_observed_cik,025816109,"(025816109,)",same_observed_full_cusip,False,1104097,False,False
12,001449,1422815,"Aflac, Inc.","(AFLAC INC,)",0000004977,"(4977,)",same_observed_cik,001055102,"(001055102,)",same_observed_full_cusip,False,1422815,False,False


### Findings

- Direct GVKEY matching identifies 539 of 652 candidate firms (82.67%), each with one RCID; no RCID is shared across sample GVKEY. The 113 unmatched firms describe gaps in the ticker-filtered extract, not necessarily in the full Revelio database.
- All 539 direct candidates have comparable CIK. CIK agrees for 511 and conflicts for 28 (5.19%); full CUSIP agrees for 537 and differs for two. Four candidates report an ultimate parent different from their RCID. These discrepancies remain unresolved, and no parent substitution was applied.
- The broad review rule flags 512 direct candidates, largely because 509 names do not match after case and whitespace normalization. Cross-checking the saved conflict and hierarchy tables shows 32 distinct candidates with CIK, CUSIP, or parent concerns, including overlap between flags; the remaining 480 flagged candidates have name-only differences. Prioritize the 32 identity/hierarchy cases. Name-only flags are lower-priority review cues, not evidence of incorrect matching or automatic clearance.
- Local CIK and full-CUSIP searches identify candidates for 25 and 26 additional GVKEY, respectively. The combined methods provide candidates for 565/652 firms (86.66%), leaving 87 without any local candidate. This exceeds the 85% follow-up threshold only as candidate availability, not as an accepted matching rate; the 26 additions require review.
- No supplementary candidate has a different nonmissing Revelio GVKEY. This absence of contradiction does not establish identity when the Revelio GVKEY is missing. No supplementary candidate, conflicting identifier, or historical parent relationship has been accepted automatically, and no additional data were downloaded.

## Part 6 - Matching Rates, Exceptions, and Research Log Preparation

Run Parts 0-5 first. Recompute this report from their in-memory DataFrames without
changing source records, earlier selection rules, or candidate acceptance. PERMNO,
membership intervals, and GVKEY are different units and have separate denominators.
The GVKEY-level reporting universe comes from the P/C candidate links; unresolved
security intervals remain visible separately and are not assigned a verified firm ID.

No acceptance register or final inclusion rule has been approved. Therefore, report
candidate availability rather than an accepted crosswalk rate. A diagnostic check
passing is not equivalent to an identity match being accepted.

### 6.1 Consolidated coverage

Every row states its unit, numerator, denominator, and evidence status. The CRSP row
is a sample inventory baseline, not a cross-database match. Date-overlap coverage means
at least some shared dates; complete temporal coverage is a different requirement.
Revelio rates refer to this local ticker-filtered extract. No final accepted rate is
computed or inferred from these tables.

In [235]:
report_gvkey = set(candidate_companies["gvkey_key"])
report_permno_count = membership_for_ccm["permno_key"].nunique()
report_interval_count = len(membership_for_ccm)
report_direct_gvkey = set(direct_matches.loc[direct_matches["rcid_key"].notna(), "gvkey_key"])
report_cik_gvkey = set(cik_candidates["gvkey_key"])
report_cusip_gvkey = set(cusip_candidates["gvkey_key"])
report_any_gvkey = report_direct_gvkey | report_cik_gvkey | report_cusip_gvkey
assert report_any_gvkey.issubset(report_gvkey)

coverage_report_rows = [{
    "stage": "CRSP historical sample",
    "unit": "PERMNO", "numerator": report_permno_count,
    "denominator": report_permno_count, "status": "Observed input universe; not a match",
}]
ccm_report_stages = {
    "CCM record with GVKEY; dates ignored": candidate_links.loc[
        candidate_links["has_ccm_record"] & candidate_links["has_gvkey"]
    ],
    "CCM date overlap; all LINKPRIM": date_valid_links,
    "CCM date overlap; P/C": primary_links,
    "CCM single P/C link spans effective interval": primary_links.loc[primary_links["spans_full_interval"]],
}
for stage, links in ccm_report_stages.items():
    for unit, column, denominator in [
        ("PERMNO", "permno_key", report_permno_count),
        ("Membership interval", "membership_row", report_interval_count),
    ]:
        coverage_report_rows.append({
            "stage": stage, "unit": unit, "numerator": links[column].nunique(),
            "denominator": denominator, "status": "Candidate linkage; stated checks only",
        })
for stage, firms in [
    ("Compustat has annual observations", set(compustat_sample["gvkey_key"])),
    ("Revelio direct GVKEY candidate", report_direct_gvkey),
    ("Revelio any local candidate method", report_any_gvkey),
]:
    coverage_report_rows.append({
        "stage": stage, "unit": "GVKEY", "numerator": len(firms & report_gvkey),
        "denominator": len(report_gvkey), "status": "Candidate availability; not accepted identity",
    })
coverage_report = pd.DataFrame(coverage_report_rows)
coverage_report["coverage_pct"] = (
    100 * coverage_report["numerator"] / coverage_report["denominator"].replace(0, float("nan"))
)
with pd.option_context("display.max_columns", None, "display.max_colwidth", 65):
    display(coverage_report.round({"coverage_pct": 2}))

,stage,unit,numerator,denominator,status,coverage_pct
0,CRSP historical sample,PERMNO,664,664,Observed input universe; not a match,100.00
1,CCM record with GVKEY; dates ignored,PERMNO,664,664,Candidate linkage; stated checks only,100.00
2,CCM record with GVKEY; dates ignored,Membership interval,666,666,Candidate linkage; stated checks only,100.00
3,CCM date overlap; all LINKPRIM,PERMNO,663,664,Candidate linkage; stated checks only,99.85
4,CCM date overlap; all LINKPRIM,Membership interval,665,666,Candidate linkage; stated checks only,99.85
5,CCM date overlap; P/C,PERMNO,657,664,Candidate linkage; stated checks only,98.95
6,CCM date overlap; P/C,Membership interval,659,666,Candidate linkage; stated checks only,98.95
7,CCM single P/C link spans effective interval,PERMNO,656,664,Candidate linkage; stated checks only,98.80
8,CCM single P/C link spans effective interval,Membership interval,658,666,Candidate linkage; stated checks only,98.80
9,Compustat has annual observations,GVKEY,640,652,Candidate availability; not accepted identity,98.16


### 6.2 Matching methods and unresolved status

Assign each GVKEY to exactly one method category: direct GVKEY; additional CIK only;
additional full CUSIP only; additional CIK and CUSIP; or no local candidate. A separate
method-overlap table includes direct firms with supplementary evidence. Shared firm
coverage does not necessarily mean the methods propose the same RCID.

This implementation is **GVKEY-first candidate discovery with CIK/CUSIP checks**, not
the faculty guide's CIK-first acceptance hierarchy. Reordering acceptance precedence
and handling disagreements require researcher approval. No candidate is accepted here.
Unresolved status is a separate dimension, not another method category to add to totals.

In [236]:
firm_method_report = candidate_companies[["gvkey_key"]].copy()
firm_method_report["direct_gvkey"] = firm_method_report["gvkey_key"].isin(report_direct_gvkey)
firm_method_report["cik_candidate"] = firm_method_report["gvkey_key"].isin(report_cik_gvkey)
firm_method_report["cusip_candidate"] = firm_method_report["gvkey_key"].isin(report_cusip_gvkey)
firm_method_report["has_any_candidate"] = firm_method_report[[
    "direct_gvkey", "cik_candidate", "cusip_candidate",
]].any(axis=1)
firm_method_report["method_category"] = "No local candidate"
no_direct = ~firm_method_report["direct_gvkey"]
firm_method_report.loc[
    no_direct & firm_method_report["cik_candidate"] & ~firm_method_report["cusip_candidate"],
    "method_category",
] = "Additional CIK only"
firm_method_report.loc[
    no_direct & ~firm_method_report["cik_candidate"] & firm_method_report["cusip_candidate"],
    "method_category",
] = "Additional full CUSIP only"
firm_method_report.loc[
    no_direct & firm_method_report["cik_candidate"] & firm_method_report["cusip_candidate"],
    "method_category",
] = "Additional CIK and full CUSIP"
firm_method_report.loc[firm_method_report["direct_gvkey"], "method_category"] = "Direct GVKEY candidate"
method_breakdown = firm_method_report.groupby("method_category").size().rename("firms").to_frame()
assert method_breakdown["firms"].sum() == len(report_gvkey)
display(method_breakdown)
display(firm_method_report.groupby(
    ["direct_gvkey", "cik_candidate", "cusip_candidate"],
).size().rename("firms_in_each_overlap_pattern").to_frame())

,firms
method_category,
Additional CIK and full CUSIP,25
Additional full CUSIP only,1
Direct GVKEY candidate,539
No local candidate,87


firms_in_each_overlap_pattern
direct_gvkey cik_candidate cusip_candidate                               
False        False         False                                       87
                           True                                         1
             True          True                                        25
True         False         False                                        2
                           True                                        26
             True          True                                       511

### 6.3 Exception summary without double-counting firms

Keep CCM interval exceptions at the interval/security level. Do not force unresolved
J-link or boundary cases into the P/C-derived GVKEY universe. Within that GVKEY universe,
use one row per firm and Boolean flags; the union counts each firm once even when it
has several concerns. Individual flag counts are not additive.

Distinguish CIK/CUSIP disagreements and hierarchy differences from name-only review
cues. Name-only differences may be formatting differences, but are not declared harmless
without review. Supplemental candidates remain unreviewed even when no conflict flag
is present. No priority label is an acceptance or rejection decision.

In [237]:
uncovered_pc_intervals = set(membership_for_ccm["membership_row"]) - set(primary_links["membership_row"])
j_only_interval_ids = set(date_valid_links.loc[
    date_valid_links["linkprim_key"].eq("J"), "membership_row",
]) & uncovered_pc_intervals
boundary_interval_ids = set(one_day_boundaries["membership_row"]) & uncovered_pc_intervals
partial_interval_ids = set(primary_links["membership_row"]) - set(
    primary_links.loc[primary_links["spans_full_interval"], "membership_row"]
)
ccm_interval_review = membership_for_ccm.copy()
ccm_interval_review["j_link_without_pc"] = ccm_interval_review["membership_row"].isin(j_only_interval_ids)
ccm_interval_review["one_day_boundary_without_pc"] = ccm_interval_review["membership_row"].isin(boundary_interval_ids)
ccm_interval_review["pc_without_single_full_interval_link"] = ccm_interval_review["membership_row"].isin(partial_interval_ids)
ccm_interval_review["no_date_overlapping_pc"] = ccm_interval_review["membership_row"].isin(uncovered_pc_intervals)
ccm_report_flags = [
    "j_link_without_pc", "one_day_boundary_without_pc",
    "pc_without_single_full_interval_link", "no_date_overlapping_pc",
]
ccm_exception_counts = []
for flag in ccm_report_flags:
    flagged_rows = ccm_interval_review.loc[ccm_interval_review[flag]]
    ccm_exception_counts.append({
        "exception": flag, "membership_intervals": len(flagged_rows),
        "distinct_permno": flagged_rows["permno_key"].nunique(),
    })
display(pd.DataFrame(ccm_exception_counts))
print("Distinct CCM intervals with any listed concern:", int(ccm_interval_review[ccm_report_flags].any(axis=1).sum()))
# No single full-interval link does not by itself prove a gap if several links could jointly cover it.

,exception,membership_intervals,distinct_permno
0,j_link_without_pc,6,6
1,one_day_boundary_without_pc,1,1
2,pc_without_single_full_interval_link,1,1
3,no_date_overlapping_pc,7,7


Distinct CCM intervals with any listed concern: 8


In [238]:
firm_exceptions = firm_method_report.copy()
firm_exceptions["compustat_absent"] = ~firm_exceptions["gvkey_key"].isin(compustat_sample["gvkey_key"])
firm_exceptions["partial_pc_interval"] = firm_exceptions["gvkey_key"].isin(
    primary_links.loc[primary_links["membership_row"].isin(partial_interval_ids), "gvkey_key"]
)
firm_exceptions["cik_conflict"] = firm_exceptions["gvkey_key"].isin(cik_conflicts["gvkey_key"])
firm_exceptions["cusip_disagreement"] = firm_exceptions["gvkey_key"].isin(
    direct_matches.loc[direct_matches["cusip_status"].eq("different_observed_cusip_review"), "gvkey_key"]
)
firm_exceptions["parent_rcid_difference"] = firm_exceptions["gvkey_key"].isin(
    direct_matches.loc[direct_matches["different_ultimate_parent"], "gvkey_key"]
)
non_name_flags = [flag for flag in revelio_review_flags if flag != "name_needs_review"]
firm_exceptions["direct_non_name_concern"] = firm_exceptions["gvkey_key"].isin(
    direct_matches.loc[direct_matches[non_name_flags].any(axis=1), "gvkey_key"]
)
firm_exceptions["name_only_review"] = (
    firm_exceptions["gvkey_key"].isin(direct_matches.loc[direct_matches["name_needs_review"], "gvkey_key"])
    & ~firm_exceptions["direct_non_name_concern"]
)
firm_exceptions["additional_unreviewed_candidate"] = (
    ~firm_exceptions["direct_gvkey"] & firm_exceptions["has_any_candidate"]
)
firm_exceptions["no_local_candidate"] = ~firm_exceptions["has_any_candidate"]
firm_exceptions["supplementary_gvkey_conflict"] = firm_exceptions["gvkey_key"].isin(
    pd.concat([
        cik_candidates.loc[cik_candidates["different_nonmissing_revelio_gvkey"], "gvkey_key"],
        cusip_candidates.loc[cusip_candidates["different_nonmissing_revelio_gvkey"], "gvkey_key"],
    ])
)
all_candidate_pairs = pd.concat([
    direct_matches.loc[direct_matches["rcid_key"].notna(), ["gvkey_key", "rcid_key"]],
    cik_candidates[["gvkey_key", "rcid_key"]], cusip_candidates[["gvkey_key", "rcid_key"]],
], ignore_index=True)
rcids_by_firm = all_candidate_pairs.groupby("gvkey_key")["rcid_key"].nunique()
firm_exceptions["multiple_rcid_across_methods"] = firm_exceptions["gvkey_key"].map(rcids_by_firm).fillna(0).gt(1)
firm_exception_flags = [
    "compustat_absent", "partial_pc_interval", "cik_conflict", "cusip_disagreement",
    "parent_rcid_difference", "direct_non_name_concern", "name_only_review",
    "additional_unreviewed_candidate", "no_local_candidate",
    "supplementary_gvkey_conflict", "multiple_rcid_across_methods",
]
display(firm_exceptions[firm_exception_flags].sum().rename("distinct_gvkey").to_frame())
print("GVKEY with any listed concern (union, not sum):", int(firm_exceptions[firm_exception_flags].any(axis=1).sum()))
print("GVKEY with CIK, CUSIP, or parent concerns (union):", int(firm_exceptions[
    ["cik_conflict", "cusip_disagreement", "parent_rcid_difference"]
].any(axis=1).sum()))

,distinct_gvkey
compustat_absent,12
partial_pc_interval,1
cik_conflict,28
cusip_disagreement,2
parent_rcid_difference,4
direct_non_name_concern,32
name_only_review,480
additional_unreviewed_candidate,26
no_local_candidate,87
supplementary_gvkey_conflict,0


GVKEY with any listed concern (union, not sum): 625
GVKEY with CIK, CUSIP, or parent concerns (union): 32


In [239]:
firm_exceptions["review_status"] = "No local candidate"
firm_exceptions.loc[firm_exceptions["additional_unreviewed_candidate"], "review_status"] = "Supplementary candidate; unresolved"
firm_exceptions.loc[firm_exceptions["direct_gvkey"], "review_status"] = "Direct candidate; not yet accepted"
firm_exceptions.loc[
    firm_exceptions["direct_gvkey"] & firm_exceptions["name_only_review"], "review_status",
] = "Direct candidate; name-only cue"
priority_review = (
    firm_exceptions["direct_non_name_concern"]
    | firm_exceptions["supplementary_gvkey_conflict"]
    | firm_exceptions["multiple_rcid_across_methods"]
)
firm_exceptions.loc[
    firm_exceptions["direct_gvkey"] & priority_review, "review_status",
] = "Direct candidate; identity or hierarchy unresolved"
review_status_counts = firm_exceptions.groupby("review_status").size().rename("distinct_gvkey").to_frame()
assert review_status_counts["distinct_gvkey"].sum() == len(report_gvkey)
display(review_status_counts)
print("Firms with any unresolved candidate (not an additional additive category):", int(firm_exceptions["has_any_candidate"].sum()))

,distinct_gvkey
review_status,
Direct candidate; identity or hierarchy unresolved,32
Direct candidate; name-only cue,480
Direct candidate; not yet accepted,27
No local candidate,87
Supplementary candidate; unresolved,26


Firms with any unresolved candidate (not an additional additive category): 565


### 6.4 Provisional employment ranking of unmatched firms

Prepare two lists: firms without a direct Revelio GVKEY candidate, and firms without
any candidate from current local methods. Both lists use the P/C-derived GVKEY universe;
securities without a resolved P/C firm mapping cannot be ranked as verified companies.

For each GVKEY, use the latest **reporting date with numeric EMP**, displaying its fiscal
year(s). Compustat EMP is generally measured in thousands of employees; retain those
source units and do not multiply values. This is a provisional ranking aid, not an
approved annual reference record. The latest available value may predate sample end
or fall outside the firm's index-membership period.

Keep every source row at the selected date. If tied records agree on EMP, use that shared
value for ranking and display their formats and fiscal years. If they disagree, retain
the range and put the firm in a separate unresolved list; do not choose INDL, FS, a
maximum, or a minimum as authoritative. Missing/undated EMP also remains unranked.
Accordingly, a top-ten table is restricted to rankable firms; it is not a complete
ordering of all unmatched firms when unresolved cases exist. Equal EMP values are
ordered by GVKEY for display only; report any tie at the tenth position.

These are licensed record-level outputs for local review. Review and clear such outputs
before any future GitHub upload. No files are exported by these cells.

In [240]:
emp_ranking_candidates = compustat_sample.loc[
    compustat_sample["emp_numeric"].notna() & compustat_sample["report_date"].notna()
].copy()
latest_numeric_date = emp_ranking_candidates.groupby("gvkey_key")["report_date"].transform("max")
latest_emp_records = emp_ranking_candidates.loc[
    emp_ranking_candidates["report_date"].eq(latest_numeric_date)
].copy()
latest_emp_summary = latest_emp_records.groupby("gvkey_key").agg(
    selected_report_date=("report_date", "max"),
    fiscal_years=("fyear_key", lambda values: tuple(sorted(values.dropna().unique()))),
    formats=("indfmt_key", lambda values: tuple(sorted(values.dropna().unique()))),
    source_rows_at_date=("compustat_row", "size"),
    distinct_emp_values=("emp_numeric", "nunique"),
    emp_min_thousands=("emp_numeric", "min"),
    emp_max_thousands=("emp_numeric", "max"),
).reset_index()
latest_emp_summary["ranking_emp_thousands"] = latest_emp_summary["emp_min_thousands"].where(
    latest_emp_summary["distinct_emp_values"].eq(1)
)
latest_emp_summary["conflicting_latest_emp"] = latest_emp_summary["distinct_emp_values"].gt(1)
latest_emp_summary["multiple_fiscal_year_labels"] = latest_emp_summary["fiscal_years"].map(len).gt(1)

employment_ranking = firm_method_report.merge(
    company_evidence[["gvkey_key", "observed_names"]], on="gvkey_key", how="left", validate="one_to_one",
).merge(latest_emp_summary, on="gvkey_key", how="left", validate="one_to_one")
employment_ranking["rankable"] = employment_ranking["ranking_emp_thousands"].notna()
employment_ranking["ranking_issue"] = "No dated numeric EMP available"
employment_ranking.loc[employment_ranking["rankable"], "ranking_issue"] = "Rankable; provisional shared EMP value"
employment_ranking.loc[
    employment_ranking["conflicting_latest_emp"].fillna(False), "ranking_issue",
] = "Conflicting EMP at latest numeric reporting date"
print("Source rows with numeric EMP but unparseable/missing reporting date:", int((
    compustat_sample["emp_numeric"].notna() & compustat_sample["report_date"].isna()
).sum()))

Source rows with numeric EMP but unparseable/missing reporting date: 0


In [241]:
unmatched_ranking_scopes = {
    "No direct GVKEY candidate": employment_ranking.loc[~employment_ranking["direct_gvkey"]],
    "No candidate under any local method": employment_ranking.loc[~employment_ranking["has_any_candidate"]],
}
top_unmatched_by_scope = {}
unranked_by_scope = {}
ranking_diagnostics = []
ranking_columns = [
    "gvkey_key", "observed_names", "selected_report_date", "fiscal_years", "formats",
    "ranking_emp_thousands", "multiple_fiscal_year_labels",
]
for scope, firms in unmatched_ranking_scopes.items():
    rankable_firms = firms.loc[firms["rankable"]].sort_values(
        ["ranking_emp_thousands", "gvkey_key"], ascending=[False, True],
    ).copy()
    rankable_firms["employment_rank"] = rankable_firms["ranking_emp_thousands"].rank(
        ascending=False, method="min",
    ).astype("int64")
    top_unmatched_by_scope[scope] = rankable_firms.head(10).copy()
    unranked_by_scope[scope] = firms.loc[~firms["rankable"]].copy()
    cutoff_tie_count = 0
    if len(rankable_firms) >= 10:
        cutoff_emp = rankable_firms.iloc[9]["ranking_emp_thousands"]
        cutoff_tie_count = int(rankable_firms["ranking_emp_thousands"].eq(cutoff_emp).sum())
    ranking_diagnostics.append({
        "scope": scope, "total_firms": len(firms), "rankable_firms": len(rankable_firms),
        "conflicting_latest_emp": int(firms["conflicting_latest_emp"].fillna(False).sum()),
        "no_dated_numeric_emp": int(firms["selected_report_date"].isna().sum()),
        "firms_at_tenth_position_emp": cutoff_tie_count,
    })
    print(f"{scope}: provisional top ten among rankable firms")
    with pd.option_context("display.max_columns", None, "display.max_colwidth", 65):
        display(top_unmatched_by_scope[scope][["employment_rank"] + ranking_columns])
    print(f"Unranked firms: {len(unranked_by_scope[scope])}; preview limit: 30")
    display(unranked_by_scope[scope][[
        "gvkey_key", "observed_names", "selected_report_date", "fiscal_years",
        "emp_min_thousands", "emp_max_thousands", "ranking_issue",
    ]].head(30))
display(pd.DataFrame(ranking_diagnostics))

No direct GVKEY candidate: provisional top ten among rankable firms


,employment_rank,gvkey_key,observed_names,selected_report_date,fiscal_years,formats,ranking_emp_thousands,multiple_fiscal_year_labels
30,1,002176,"(BERKSHIRE HATHAWAY,)",2025-12-31,"(2025,)","(FS, INDL)",387.815,False
254,2,011264,"(WALGREENS BOOTS ALLIANCE INC,)",2024-08-31,"(2024,)","(INDL,)",312.0,False
522,3,118122,"(APTIV PLC,)",2025-12-31,"(2025,)","(INDL,)",140.0,False
241,4,010795,"(UNITED AIRLINES HOLDINGS INC,)",2025-12-31,"(2025,)","(INDL,)",113.2,False
303,5,014489,"(DELL TECHNOLOGIES INC,)",2025-01-31,"(2024,)","(INDL,)",108.0,False
649,6,283184,"(SMURFIT WESTROCK PLC,)",2025-12-31,"(2025,)","(INDL,)",97.0,False
155,7,007065,"(MARSH,)",2025-12-31,"(2025,)","(INDL,)",95.0,False
100,8,004990,"(GAP INC,)",2025-01-31,"(2024,)","(INDL,)",82.0,False
616,9,177267,"(TE CONNECTIVITY PLC,)",2025-09-30,"(2025,)","(INDL,)",80.0,False
464,10,045169,"(GE VERNOVA INC,)",2025-12-31,"(2025,)","(INDL,)",78.0,False


Unranked firms: 13; preview limit: 30


,gvkey_key,observed_names,selected_report_date,fiscal_years,emp_min_thousands,emp_max_thousands,ranking_issue
4,001177,(),NaT,NaN,<NA>,<NA>,No dated numeric EMP available
211,009445,(),NaT,NaN,<NA>,<NA>,No dated numeric EMP available
232,010466,(),NaT,NaN,<NA>,<NA>,No dated numeric EMP available
350,024318,(),NaT,NaN,<NA>,<NA>,No dated numeric EMP available
362,025056,(),NaT,NaN,<NA>,<NA>,No dated numeric EMP available
369,025356,(),NaT,NaN,<NA>,<NA>,No dated numeric EMP available
374,026304,(),NaT,NaN,<NA>,<NA>,No dated numeric EMP available
384,028119,"(BROOKFIELD PROPERTY REIT,)",NaT,NaN,<NA>,<NA>,No dated numeric EMP available
402,029173,(),NaT,NaN,<NA>,<NA>,No dated numeric EMP available
503,065967,(),NaT,NaN,<NA>,<NA>,No dated numeric EMP available


No candidate under any local method: provisional top ten among rankable firms


,employment_rank,gvkey_key,observed_names,selected_report_date,fiscal_years,formats,ranking_emp_thousands,multiple_fiscal_year_labels
30,1,002176,"(BERKSHIRE HATHAWAY,)",2025-12-31,"(2025,)","(FS, INDL)",387.815,False
254,2,011264,"(WALGREENS BOOTS ALLIANCE INC,)",2024-08-31,"(2024,)","(INDL,)",312.0,False
155,3,007065,"(MARSH,)",2025-12-31,"(2025,)","(INDL,)",95.0,False
100,4,004990,"(GAP INC,)",2025-01-31,"(2024,)","(INDL,)",82.0,False
207,5,008972,"(RAYTHEON CO,)",2019-12-31,"(2019,)","(INDL,)",70.0,False
410,6,029830,"(WESTROCK CO,)",2023-09-30,"(2023,)","(INDL,)",56.1,False
176,7,007922,"(NORDSTROM INC,)",2025-01-31,"(2024,)","(INDL,)",55.0,False
130,8,006136,"(INTERPUBLIC GROUP OF COS,)",2024-12-31,"(2024,)","(INDL,)",53.3,False
25,9,002019,"(BANK OF NEW YORK MELLON CORP,)",2025-12-31,"(2025,)","(FS, INDL)",48.1,False
263,10,011584,"(FOOT LOCKER INC,)",2025-01-31,"(2024,)","(INDL,)",47.306,False


Unranked firms: 13; preview limit: 30


,gvkey_key,observed_names,selected_report_date,fiscal_years,emp_min_thousands,emp_max_thousands,ranking_issue
4,001177,(),NaT,NaN,<NA>,<NA>,No dated numeric EMP available
211,009445,(),NaT,NaN,<NA>,<NA>,No dated numeric EMP available
232,010466,(),NaT,NaN,<NA>,<NA>,No dated numeric EMP available
350,024318,(),NaT,NaN,<NA>,<NA>,No dated numeric EMP available
362,025056,(),NaT,NaN,<NA>,<NA>,No dated numeric EMP available
369,025356,(),NaT,NaN,<NA>,<NA>,No dated numeric EMP available
374,026304,(),NaT,NaN,<NA>,<NA>,No dated numeric EMP available
384,028119,"(BROOKFIELD PROPERTY REIT,)",NaT,NaN,<NA>,<NA>,No dated numeric EMP available
402,029173,(),NaT,NaN,<NA>,<NA>,No dated numeric EMP available
503,065967,(),NaT,NaN,<NA>,<NA>,No dated numeric EMP available


,scope,total_firms,rankable_firms,conflicting_latest_emp,no_dated_numeric_emp,firms_at_tenth_position_emp
0,No direct GVKEY candidate,113,100,0,13,1
1,No candidate under any local method,87,74,0,13,1


### 6.5 Research-log evidence and pending decisions

Prepare source provenance below for the preliminary log. A date encoded in a filename
is reported as such, not independently verified as the actual WRDS retrieval timestamp.
The Compustat filename has no date; its pull date is not documented here. Do not infer
it from local file modification time. The existing membership CSV is an intermediate
input, while the two TXT files are WRDS query aids.

After the researcher executes and saves Part 6, inspect the actual saved outputs before
writing the authorized `data/clean/crosswalk_log.md`. The log will summarize the study
window, source inventory, identifier units, CCM overlap/E-marker rules, candidate
methods, coverage, exceptions, and decisions. It will note that historical tickers and
undated parent relationships do not establish historical entity identity. It must state
that no unreviewed candidates were accepted and no final crosswalk was exported.

**Recorded procedures:** preserve source identifiers; use inclusive date intersections;
retain `E` and use the study end for its effective end; preserve all formats; distinguish
candidate availability from acceptance; rank unmatched firms only provisionally.

**Pending decisions:** CCM J/boundary/partial coverage; conflicting identities and parents;
supplementary RCID acceptance; CIK-first acceptance precedence; company reference and
INDL/FS rules; fiscal-year/sample rules; missing-data treatment; and any ranking ties
with conflicting EMP. No new acceptance rule is implemented here.

In [242]:
source_log_rows = []
for source, source_path in input_paths.items():
    filename_token = source_path.stem.rsplit("_", 1)[-1]
    filename_date = pd.NaT
    if len(filename_token) == 8 and filename_token.isdigit():
        filename_date = pd.to_datetime(filename_token, format="%Y%m%d", errors="coerce")
    source_log_rows.append({
        "source": source,
        "filename": source_path.relative_to(project_root).as_posix(),
        "date_encoded_in_filename": filename_date,
        "pull_date_evidence": "Filename date; actual pull not independently verified" if pd.notna(filename_date) else "Not documented",
        "input_rows": len(datasets[source]),
        "source_columns": len(source_columns[source]),
    })
source_log_inventory = pd.DataFrame(source_log_rows)
print(f"Research window: {study_start.date()} to {study_end.date()}")
with pd.option_context("display.max_columns", None, "display.max_colwidth", 85):
    display(source_log_inventory)
print("Intermediate inputs:")
for name, intermediate_path in intermediate_paths.items():
    print(name, intermediate_path.relative_to(project_root).as_posix())
print("Research log: pending review of saved Part 6 outputs; no file written by this cell.")

Research window: 2018-01-01 to 2025-12-31


,source,filename,date_encoded_in_filename,pull_date_evidence,input_rows,source_columns
0,crsp,data/raw/crsp/crsp_sp500_20261022.csv,2026-10-22,Filename date; actual pull not independently verified,1013723,42
1,ccm,data/raw/ccm/ccm_link_history_20261023.csv,2026-10-23,Filename date; actual pull not independently verified,788,8
2,compustat,data/raw/compustat/Compustat Fundamentals Annual.csv,NaT,Not documented,107179,15
3,revelio,data/raw/revelio/revelio_company_mapping_20261023.csv,2026-10-23,Filename date; actual pull not independently verified,1124,16


Intermediate inputs:
membership_intervals data/clean/sp500_membership_intervals.csv
permno_query_aid data/clean/sp500_permno_for_ccm.txt
ticker_query_aid data/clean/sp500_tickers_for_revelio.txt
Research log: pending review of saved Part 6 outputs; no file written by this cell.


### Findings

- P/C date-overlap coverage is 657/664 securities and 659/666 membership intervals (98.95% in each case). Six J-link intervals, one one-day boundary case, and one partially covered P/C interval remain unresolved.
- Compustat covers 640/652 candidate GVKEY (98.16%). Direct Revelio GVKEY candidates cover 539/652 (82.67%); supplementary evidence raises candidate availability to 565/652 (86.66%). These are candidate rates, not accepted identity-match rates.
- The mutually exclusive Revelio categories are 539 direct candidates, 25 additional firms with both CIK and full-CUSIP candidates, one additional full-CUSIP-only firm, and 87 firms without a local candidate. No firm has multiple RCID across the current methods.
- Among direct candidates, 32 firms have identity or hierarchy concerns, 480 have name-only cues, and 27 have no listed review flags. None has been automatically accepted. Overlapping exception flags must not be summed as independent firms.
- Provisional employment rankings cover 100 of 113 firms without a direct candidate and 74 of 87 firms without any local candidate. Each scope excludes 13 firms without dated numeric EMP. There are no conflicting EMP values at the selected latest dates and no tie at the tenth position. The lists do not establish the largest firms among those lacking employment data.

## Part 7 - Provisional Crosswalk and Complete Membership Bridge

### 7.1 Approved provisional design and approval record

The researcher approved the provisional design and researcher-controlled export, not
individual RCID or ambiguous historical identities. Run 7.1-7.5, including the new
bridge and quality checks, before manually executing 7.6.

- Main table: one row per P/C-candidate GVKEY; provisional key `gvkey:<source GVKEY>`.
- Scalar reference values use unique observed values only; no arbitrary PERMNO or
  authoritative INDL/FS record is selected. Missing historical dates/scores remain null.
- Main JSON retains all P/C histories. The authorized membership bridge retains every
  original interval, including J-only and boundary cases, with full nested CCM evidence.
- Candidate assignments are not acceptance. RCID remains null without a researcher
  decision; parent RCID remains snapshot evidence, not a historical assertion.
- Multiple intervals are not compressed into one span. Early flagged starts and
  sample-end exits remain null in main scalar fields and visible in source history.

All four design flags and manual export are approved. Individual identity decisions
remain pending; the empty decision table must not be populated automatically.

In [ ]:
import importlib.util
import json

# Approved for provisional design and researcher-controlled export only.
# Complete membership bridge preservation is authorized.
# These flags do not accept any individual RCID, LS/J link, or disputed identity.
export_approvals = {
    "candidate_gvkey_scope_and_firm_id": True,
    "unique_observed_reference_values": True,
    "embedded_history_and_nullable_dates": True,
    "provisional_unresolved_statuses": True,
}
export_authorized = True

# Enter only reviewed decisions: one RCID per GVKEY, with a written justification.
approved_rcid_decisions = pd.DataFrame(
    columns=["gvkey_key", "rcid_key", "decision_basis"], dtype="string",
)
print("pyarrow available in this kernel:", importlib.util.find_spec("pyarrow") is not None)
display(pd.Series(export_approvals, name="approved").to_frame())

### 7.2 Provisional scalar fields

Build a draft only. Group observed reference values without selecting a preferred
annual row. Identifiers remain strings. Conflicting or multiple values produce null
scalar fields, while earlier tables retain the evidence. Method and review statuses
remain pre-adjudication diagnostics, even after an explicit RCID decision is entered.

In [244]:
reference_evidence = compustat_identity_evidence.copy()
reference_evidence["cusip8_observed"] = reference_evidence["cusip_for_comparison"].str[:8]
reference_sets = reference_evidence.groupby("gvkey_key").agg(
    cik_values=("cik_for_comparison", lambda values: tuple(sorted(values.dropna().unique()))),
    cusip8_values=("cusip8_observed", lambda values: tuple(sorted(values.dropna().unique()))),
    ticker_values=("tic_key", lambda values: tuple(sorted(values.dropna().unique()))),
).reset_index()
crosswalk_draft = firm_exceptions.merge(
    reference_sets, on="gvkey_key", how="left", validate="one_to_one",
).copy()
for source, target in [("cik_values", "cik"), ("cusip8_values", "cusip8"), ("ticker_values", "ticker")]:
    crosswalk_draft[target] = pd.array([
        values[0] if isinstance(values, tuple) and len(values) == 1 else pd.NA
        for values in crosswalk_draft[source]
    ], dtype="string")
permno_sets = primary_links.groupby("gvkey_key")["permno_key"].agg(
    lambda values: tuple(sorted(values.dropna().unique()))
)
crosswalk_draft["permno"] = pd.array([
    permno_sets[key][0] if len(permno_sets[key]) == 1 else pd.NA
    for key in crosswalk_draft["gvkey_key"]
], dtype="string")
crosswalk_draft["gvkey"] = crosswalk_draft["gvkey_key"].astype("string")
crosswalk_draft["firm_id"] = "gvkey:" + crosswalk_draft["gvkey"]
crosswalk_draft["artifact_status"] = "provisional_candidate_crosswalk"
crosswalk_draft["rcid_status"] = "unreviewed_candidate"
crosswalk_draft.loc[~crosswalk_draft["has_any_candidate"], "rcid_status"] = "no_local_candidate"
for column in ["rcid", "ultimate_parent_rcid", "transcript_symbol", "match_method", "decision_basis"]:
    crosswalk_draft[column] = pd.Series(pd.NA, index=crosswalk_draft.index, dtype="string")
crosswalk_draft["match_score"] = pd.Series(pd.NA, index=crosswalk_draft.index, dtype="Float64")
for column in ["ticker_valid_from", "ticker_valid_to", "sp500_entry", "sp500_exit"]:
    crosswalk_draft[column] = pd.Series(pd.NaT, index=crosswalk_draft.index, dtype="datetime64[ns]")

### 7.3 Preserve dated relationships and candidate evidence

Embed one history entry per P/C candidate link, retaining all securities and membership
intervals. The scalar membership dates never summarize several intervals as one span.
Candidate RCID histories preserve evidence methods without accepting those candidates.
These JSON fields are local audit data and are not printed as large record-level outputs.

In [245]:
historical_link_rows = []
for gvkey, links in primary_links.groupby("gvkey_key"):
    history = []
    for row in links.itertuples(index=False):
        history.append({
            "membership_row": int(row.membership_row), "ccm_row": int(row.ccm_row),
            "permno": row.permno, "permco": row.permco,
            "membership_start_source": row.mbrstartdt, "membership_end_source": row.mbrenddt,
            "link_start_source": row.linkdt, "link_end_source": row.linkenddt,
            "overlap_start": row.overlap_start.strftime("%Y-%m-%d"),
            "overlap_end": row.overlap_end.strftime("%Y-%m-%d"),
            "linkprim": row.linkprim, "linktype": row.linktype,
            "spans_full_interval": bool(row.spans_full_interval),
        })
    historical_link_rows.append({
        "gvkey": gvkey, "historical_links_json": json.dumps(history, ensure_ascii=False),
        "membership_interval_count": links["membership_row"].nunique(),
        "linked_permno_count": links["permno_key"].nunique(),
    })
    if links["membership_row"].nunique() == 1:
        # Every candidate for this one membership_row has the same source interval.
        source_interval = links.iloc[0]
        original_start = pd.to_datetime(source_interval["mbrstartdt"], format="%Y-%m-%d", errors="coerce")
        original_end = pd.to_datetime(source_interval["mbrenddt"], format="%Y-%m-%d", errors="coerce")
        firm_mask = crosswalk_draft["gvkey"].eq(gvkey)
        if pd.notna(original_start) and original_start > pd.Timestamp("1925-12-31"):
            crosswalk_draft.loc[firm_mask, "sp500_entry"] = original_start
        if pd.notna(original_end) and original_end != study_end:
            crosswalk_draft.loc[firm_mask, "sp500_exit"] = original_end
crosswalk_draft = crosswalk_draft.merge(
    pd.DataFrame(historical_link_rows), on="gvkey", how="left", validate="one_to_one",
)

In [246]:
candidate_evidence_frames = []
for method, candidates in [
    ("direct_gvkey", direct_matches), ("cik", cik_candidates), ("full_cusip", cusip_candidates),
]:
    evidence = candidates.loc[candidates["rcid_key"].notna(), ["gvkey_key", "rcid_key"]].copy()
    evidence["method"] = method
    candidate_evidence_frames.append(evidence)
candidate_evidence = pd.concat(candidate_evidence_frames, ignore_index=True)
# Group repeated evidence occurrences; preserve every distinct RCID and evidence method.
candidate_method_sets = candidate_evidence.groupby(["gvkey_key", "rcid_key"])["method"].agg(
    lambda values: tuple(sorted(values.unique()))
).reset_index()
candidate_history = {}
for gvkey, rows in candidate_method_sets.groupby("gvkey_key"):
    candidate_history[gvkey] = json.dumps([
        {"rcid": row.rcid_key, "methods": list(row.method)} for row in rows.itertuples(index=False)
    ])
crosswalk_draft["revelio_candidates_json"] = crosswalk_draft["gvkey"].map(candidate_history).fillna("[]")

### 7.4 Explicit RCID decisions and the intended key

Every decision must identify an existing local candidate and include a nonblank basis.
No decision is generated from passing checks, name similarity, or a score. Explain any
remaining identifier or hierarchy concerns in the decision basis. Acceptance concerns
entity identity only; historical time validity remains unverified. Existing exception
flags are retained, not silently cleared. The proposed firm key is checked for uniqueness.

In [247]:
rcid_decisions = approved_rcid_decisions.copy()
for column in rcid_decisions.columns:
    rcid_decisions[column] = rcid_decisions[column].astype("string").str.strip().replace("", pd.NA)
if rcid_decisions.isna().any(axis=None):
    raise ValueError("Every decision requires GVKEY, RCID, and a nonblank decision basis.")
if rcid_decisions["gvkey_key"].duplicated().any():
    raise ValueError("Conflicting or repeated RCID decisions for a GVKEY require review.")
checked_decisions = rcid_decisions.merge(
    candidate_method_sets, on=["gvkey_key", "rcid_key"], how="left", validate="one_to_one", indicator=True,
)
if not checked_decisions["_merge"].eq("both").all():
    raise ValueError("A decision does not correspond to an existing local RCID candidate.")

for decision in checked_decisions.itertuples(index=False):
    firm_mask = crosswalk_draft["gvkey"].eq(decision.gvkey_key)
    crosswalk_draft.loc[firm_mask, "rcid"] = decision.rcid_key
    crosswalk_draft.loc[firm_mask, "rcid_status"] = "researcher_accepted_identity_time_unverified"
    crosswalk_draft.loc[firm_mask, "match_method"] = "+".join(decision.method)
    crosswalk_draft.loc[firm_mask, "decision_basis"] = decision.decision_basis
    parent_values = revelio_for_mapping.loc[
        revelio_for_mapping["rcid_key"].eq(decision.rcid_key), "ultimate_parent_rcid_key",
    ].dropna().unique()
    if len(parent_values) == 1:
        crosswalk_draft.loc[firm_mask, "ultimate_parent_rcid"] = parent_values[0]
crosswalk_draft["parent_validity_status"] = "snapshot_only_historical_validity_unverified"

required_crosswalk_columns = [
    "firm_id", "gvkey", "cik", "cusip8", "permno", "ticker", "ticker_valid_from", "ticker_valid_to",
    "rcid", "ultimate_parent_rcid", "transcript_symbol", "match_method", "match_score", "sp500_entry", "sp500_exit",
]
status_columns = [
    "artifact_status", "rcid_status", "parent_validity_status", "decision_basis", "method_category", "review_status",
    "membership_interval_count", "linked_permno_count", "historical_links_json", "revelio_candidates_json",
] + firm_exception_flags
crosswalk = crosswalk_draft[required_crosswalk_columns + status_columns].copy()
date_columns_for_export = ["ticker_valid_from", "ticker_valid_to", "sp500_entry", "sp500_exit"]
count_columns_for_export = ["membership_interval_count", "linked_permno_count"]
for column in crosswalk.columns:
    if column in date_columns_for_export:
        crosswalk[column] = crosswalk[column].astype("datetime64[ns]")
    elif column in count_columns_for_export:
        crosswalk[column] = crosswalk[column].astype("Int64")
    elif column in firm_exception_flags:
        crosswalk[column] = crosswalk[column].astype("boolean")
    elif column != "match_score":
        crosswalk[column] = crosswalk[column].astype("string")

In [248]:
assert crosswalk["firm_id"].notna().all() and crosswalk["firm_id"].is_unique
assert crosswalk["gvkey"].notna().all() and crosswalk["gvkey"].is_unique
assert set(crosswalk["gvkey"]) == set(candidate_companies["gvkey_key"])
assert crosswalk["cusip8"].dropna().str.len().eq(8).all()
assert crosswalk.loc[crosswalk["rcid"].notna(), "decision_basis"].notna().all()
assert set(crosswalk.loc[crosswalk["rcid"].notna(), "gvkey"]) == set(rcid_decisions["gvkey_key"])
preserved_history_rows = sum(len(json.loads(value)) for value in crosswalk["historical_links_json"])
assert preserved_history_rows == len(primary_links)
for column in ["ticker_valid_from", "ticker_valid_to", "transcript_symbol", "match_score"]:
    assert crosswalk[column].isna().all()
export_diagnostics = pd.Series({
    "provisional_firm_rows": len(crosswalk),
    "unique_firm_id": crosswalk["firm_id"].nunique(),
    "explicitly_accepted_rcid_rows": int(crosswalk["rcid"].notna().sum()),
    "rows_without_accepted_rcid": int(crosswalk["rcid"].isna().sum()),
    "rows_with_multiple_securities": int(crosswalk["linked_permno_count"].gt(1).sum()),
    "rows_with_multiple_membership_intervals": int(crosswalk["membership_interval_count"].gt(1).sum()),
    "preserved_pc_history_records": preserved_history_rows,
    "unresolved_non_pc_intervals_outside_firm_universe": len(uncovered_pc_intervals),
}, name="count")
display(export_diagnostics.to_frame())
display(crosswalk[required_crosswalk_columns].isna().sum().rename("null_rows").to_frame())

,count
provisional_firm_rows,652
unique_firm_id,652
explicitly_accepted_rcid_rows,0
rows_without_accepted_rcid,652
rows_with_multiple_securities,5
rows_with_multiple_membership_intervals,7
preserved_pc_history_records,659
unresolved_non_pc_intervals_outside_firm_universe,7


,null_rows
firm_id,0
gvkey,0
cik,13
cusip8,12
permno,5
ticker,12
ticker_valid_from,652
ticker_valid_to,652
rcid,652
ultimate_parent_rcid,652


### 7.4a Focused CCM quality checks

Retain LS records and display source and overlap dates. Check competing GVKEY among
all date-overlapping candidates for the same security or company, requiring shared
dates. A competing candidate is not automatically a verified conflict.

Review PERMCO/GVKEY multiplicity with dates. Multiple PERMNO can reflect share classes;
counts alone cannot distinguish sequential reorganizations from simultaneous conflicts.
Local record previews are limited to 30 rows. No records are removed or accepted.

In [ ]:
ls_review_links = primary_links.loc[primary_links["linktype_key"].eq("LS")].copy()
ls_review_links["same_security_competing_gvkey"] = 0
ls_review_links["same_company_competing_gvkey"] = 0
ls_competing_evidence = []
for index, link in ls_review_links.iterrows():
    same_security = date_valid_links["permno_key"].eq(link["permno_key"]).fillna(False)
    same_company = (
        date_valid_links["member_permco_key"].notna()
        & date_valid_links["member_permco_key"].eq(link["member_permco_key"]).fillna(False)
    )
    competing = date_valid_links.loc[
        (same_security | same_company)
        & date_valid_links["gvkey_key"].ne(link["gvkey_key"])
        & date_valid_links["overlap_start"].le(link["overlap_end"])
        & date_valid_links["overlap_end"].ge(link["overlap_start"])
    ].copy()
    ls_review_links.loc[index, "same_security_competing_gvkey"] = competing.loc[
        competing["permno_key"].eq(link["permno_key"]), "gvkey_key",
    ].nunique()
    ls_review_links.loc[index, "same_company_competing_gvkey"] = competing.loc[
        competing["member_permco_key"].eq(link["member_permco_key"]), "gvkey_key",
    ].nunique()
    competing["ls_source_ccm_row"] = link["ccm_row"]
    ls_competing_evidence.append(competing)
ls_competing_candidates = (
    pd.concat(ls_competing_evidence, ignore_index=True) if ls_competing_evidence
    else date_valid_links.iloc[:0].copy()
)
ls_display_columns = [
    "membership_row", "permno", "permco", "gvkey", "linkprim", "linktype",
    "linkdt", "linkenddt", "mbrstartdt", "mbrenddt", "overlap_start", "overlap_end",
]
print("Date-overlapping P/C LS records:", len(ls_review_links))
with pd.option_context("display.max_columns", None):
    display(ls_review_links[ls_display_columns + [
        "same_security_competing_gvkey", "same_company_competing_gvkey",
    ]].head(30))
    display(ls_competing_candidates[ls_display_columns].head(30))

In [ ]:
permco_gvkey_counts = primary_links.groupby("member_permco_key")["gvkey_key"].nunique()
gvkey_relation_counts = primary_links.groupby("gvkey_key").agg(
    distinct_permco=("member_permco_key", "nunique"),
    distinct_permno=("permno_key", "nunique"),
    membership_intervals=("membership_row", "nunique"),
)
relationship_cases = {
    "PERMCO with multiple GVKEY": primary_links["member_permco_key"].isin(permco_gvkey_counts.loc[permco_gvkey_counts.gt(1)].index),
    "GVKEY with multiple PERMCO": primary_links["gvkey_key"].isin(gvkey_relation_counts.index[gvkey_relation_counts["distinct_permco"].gt(1)]),
    "GVKEY with multiple PERMNO": primary_links["gvkey_key"].isin(gvkey_relation_counts.index[gvkey_relation_counts["distinct_permno"].gt(1)]),
    "GVKEY with multiple membership intervals": primary_links["gvkey_key"].isin(gvkey_relation_counts.index[gvkey_relation_counts["membership_intervals"].gt(1)]),
}
for label, mask in relationship_cases.items():
    exceptions = primary_links.loc[mask, ls_display_columns].sort_values(["gvkey", "permno", "overlap_start"])
    print(label, "| distinct PERMCO:", exceptions["permco"].nunique(), "| distinct GVKEY:", exceptions["gvkey"].nunique())
    with pd.option_context("display.max_columns", None):
        display(exceptions.head(30))
crosswalk["ls_review"] = crosswalk["gvkey"].isin(ls_review_links["gvkey_key"]).astype("boolean")
crosswalk["cross_company_mapping_review"] = crosswalk["gvkey"].isin(primary_links.loc[
    relationship_cases["PERMCO with multiple GVKEY"] | relationship_cases["GVKEY with multiple PERMCO"], "gvkey_key",
]).astype("boolean")

### 7.4b Complete membership bridge

Start from every original interval, including unresolved cases. Keep one row per
membership_row and all identifier-level CCM candidates in JSON. A scalar candidate
GVKEY requires one distinct P/C candidate; J-only evidence never supplies it. Scalar
link details require exactly one P/C record, otherwise the nested evidence is used.

Separate any overlap, one full-span link, full union-of-links coverage, and partial
coverage. Adjacent inclusive calendar-date segments are contiguous. Coverage checks
do not establish company identity. Preserve LS, J-only, boundary and ambiguous cases.

In [ ]:
bridge_source_columns = [
    "membership_row", "indno", "permno", "permco", "mbrstartdt", "mbrenddt", "effective_start", "effective_end",
]
membership_bridge = membership[bridge_source_columns].copy()
for column in ["candidate_gvkey", "candidate_firm_id", "linkprim", "linktype", "linkdt", "linkenddt", "assignment_status", "unresolved_reason"]:
    membership_bridge[column] = pd.Series(pd.NA, index=membership_bridge.index, dtype="string")
for column in ["overlap_start", "overlap_end"]:
    membership_bridge[column] = pd.Series(pd.NaT, index=membership_bridge.index, dtype="datetime64[ns]")
for column in ["pc_record_count", "pc_gvkey_count"]:
    membership_bridge[column] = pd.Series(0, index=membership_bridge.index, dtype="Int64")
bridge_flags = [
    "any_date_overlap", "has_pc_overlap", "single_link_full_coverage", "pc_union_full_coverage",
    "partial_coverage", "j_only_evidence", "one_day_boundary", "ls_review", "ambiguous_pc_gvkey",
]
for column in bridge_flags:
    membership_bridge[column] = pd.Series(False, index=membership_bridge.index, dtype="boolean")
membership_bridge["ccm_candidates_json"] = pd.Series("[]", index=membership_bridge.index, dtype="string")
bridge_evidence_columns = [
    "ccm_row", "gvkey", "lpermno", "lpermco", "linkprim", "linktype", "linkdt", "linkenddt",
    "link_start", "link_end_effective", "valid_link_dates", "has_gvkey", "date_overlap",
    "overlap_start", "overlap_end", "boundary_separation_days", "spans_full_interval",
]
ccm_by_interval = {key: group for key, group in candidate_links.loc[candidate_links["has_ccm_record"]].groupby("membership_row")}
pc_by_interval = {key: group for key, group in primary_links.groupby("membership_row")}

In [ ]:
for index, interval in membership_bridge.iterrows():
    candidates = ccm_by_interval.get(interval["membership_row"], candidate_links.iloc[:0])
    pc_links = pc_by_interval.get(interval["membership_row"], primary_links.iloc[:0])
    overlapping = candidates.loc[candidates["date_overlap"] & candidates["has_gvkey"]]
    pc_keys = pc_links["gvkey_key"].unique()
    has_pc = len(pc_links) > 0
    membership_bridge.loc[index, "ccm_candidates_json"] = candidates[bridge_evidence_columns].to_json(orient="records", date_format="iso")
    membership_bridge.loc[index, "pc_record_count"] = len(pc_links)
    membership_bridge.loc[index, "pc_gvkey_count"] = len(pc_keys)
    membership_bridge.loc[index, "any_date_overlap"] = len(overlapping) > 0
    membership_bridge.loc[index, "has_pc_overlap"] = has_pc
    membership_bridge.loc[index, "single_link_full_coverage"] = bool(pc_links["spans_full_interval"].any())
    j_only = not has_pc and len(overlapping) > 0 and overlapping["linkprim_key"].eq("J").all()
    boundary = candidates["boundary_separation_days"].eq(1).any()
    ls_flag = pc_links["linktype_key"].eq("LS").any()
    membership_bridge.loc[index, ["j_only_evidence", "one_day_boundary", "ls_review", "ambiguous_pc_gvkey"]] = [bool(j_only), bool(boundary), bool(ls_flag), len(pc_keys) > 1]
    if len(pc_keys) == 1:
        membership_bridge.loc[index, "candidate_gvkey"] = pc_keys[0]
        membership_bridge.loc[index, "candidate_firm_id"] = "gvkey:" + pc_keys[0]
    if len(pc_links) == 1:
        single_link = pc_links.iloc[0]
        for column in ["linkprim", "linktype", "linkdt", "linkenddt", "overlap_start", "overlap_end"]:
            membership_bridge.loc[index, column] = single_link[column]
    full_union = False
    if has_pc:
        covered_until = interval["effective_start"] - pd.Timedelta(days=1)
        for segment in pc_links.sort_values("overlap_start").itertuples(index=False):
            if segment.overlap_start > covered_until + pd.Timedelta(days=1):
                break
            covered_until = max(covered_until, segment.overlap_end)
        full_union = covered_until >= interval["effective_end"]
    membership_bridge.loc[index, "pc_union_full_coverage"] = bool(full_union)
    membership_bridge.loc[index, "partial_coverage"] = has_pc and not full_union
    reasons = []
    if not has_pc:
        reasons.append("J-only evidence" if j_only else "No date-overlapping P/C candidate")
        if boundary:
            reasons.append("One-day boundary evidence; unchanged")
    if has_pc and not full_union:
        reasons.append("Partial temporal coverage")
    if len(pc_keys) > 1:
        reasons.append("Multiple P/C candidate GVKEY; no scalar assignment")
    if ls_flag:
        reasons.append("LS source link requires review")
    membership_bridge.loc[index, "unresolved_reason"] = "; ".join(reasons) if reasons else "No listed coverage exception; identity unverified"
    membership_bridge.loc[index, "assignment_status"] = (
        "pc_candidate_unverified" if len(pc_keys) == 1 else "ambiguous_pc_candidates" if len(pc_keys) > 1 else "no_pc_assignment_unresolved"
    )

In [ ]:
for column in ["membership_row", "pc_record_count", "pc_gvkey_count"]:
    membership_bridge[column] = membership_bridge[column].astype("Int64")
bridge_date_columns = ["effective_start", "effective_end", "overlap_start", "overlap_end"]
for column in bridge_date_columns:
    membership_bridge[column] = membership_bridge[column].astype("datetime64[ns]")
for column in membership_bridge.columns:
    if column not in bridge_date_columns + bridge_flags + ["membership_row", "pc_record_count", "pc_gvkey_count"]:
        membership_bridge[column] = membership_bridge[column].astype("string")
expected_membership_rows = len(membership)
expected_unresolved_ids = set(membership["membership_row"]) - set(primary_links["membership_row"])
assert len(membership_bridge) == expected_membership_rows
assert membership_bridge["membership_row"].is_unique and membership_bridge["membership_row"].notna().all()
assert set(membership_bridge["membership_row"]) == set(membership["membership_row"])
original_interval_columns = ["indno", "permno", "permco", "mbrstartdt", "mbrenddt"]
pd.testing.assert_frame_equal(
    membership_bridge.set_index("membership_row")[original_interval_columns].sort_index(),
    membership.set_index("membership_row")[original_interval_columns].sort_index(), check_index_type=False,
)
assert set(membership_bridge.loc[~membership_bridge["has_pc_overlap"], "membership_row"]) == expected_unresolved_ids
assert membership_bridge.loc[~membership_bridge["has_pc_overlap"], ["candidate_gvkey", "candidate_firm_id"]].isna().all(axis=None)
assert membership_bridge.loc[membership_bridge["pc_gvkey_count"].ne(1), "candidate_gvkey"].isna().all()
assert set(membership_bridge["candidate_gvkey"].dropna()).issubset(set(crosswalk["gvkey"]))
assert sum(len(json.loads(value)) for value in membership_bridge["ccm_candidates_json"]) == int(candidate_links["has_ccm_record"].sum())
print("Original intervals and bridge rows:", expected_membership_rows, len(membership_bridge))
print("Unresolved non-P/C intervals preserved:", len(expected_unresolved_ids))
display(membership_bridge[bridge_flags].sum().rename("membership_intervals").to_frame())
bridge_review = membership_bridge.loc[
    ~membership_bridge["has_pc_overlap"] | membership_bridge["partial_coverage"]
    | membership_bridge["ls_review"] | membership_bridge["ambiguous_pc_gvkey"]
]
with pd.option_context("display.max_columns", None, "display.max_colwidth", 80):
    display(bridge_review.drop(columns="ccm_candidates_json").head(30))

In [ ]:
# Check the provisional schema without accepting any identity.
assert crosswalk["firm_id"].is_unique and crosswalk["gvkey"].is_unique
assert len(crosswalk) == candidate_companies["gvkey_key"].nunique()
assert crosswalk["firm_id"].eq("gvkey:" + crosswalk["gvkey"]).all()
for column in ["firm_id", "gvkey", "cik", "cusip8", "permno", "ticker", "rcid", "ultimate_parent_rcid"]:
    assert isinstance(crosswalk[column].dtype, pd.StringDtype), column
for column in date_columns_for_export:
    assert pd.api.types.is_datetime64_any_dtype(crosswalk[column]), column
assert str(crosswalk["match_score"].dtype) == "Float64"
assert crosswalk.loc[crosswalk["linked_permno_count"].gt(1), "permno"].isna().all()
assert not crosswalk["sp500_exit"].eq(study_end).any()
assert crosswalk.loc[crosswalk["membership_interval_count"].gt(1), ["sp500_entry", "sp500_exit"]].isna().all(axis=None)
for column in ["ticker_valid_from", "ticker_valid_to", "transcript_symbol", "match_score"]:
    assert crosswalk[column].isna().all()
exported_rcid_pairs = set(zip(crosswalk.loc[crosswalk["rcid"].notna(), "gvkey"], crosswalk.loc[crosswalk["rcid"].notna(), "rcid"]))
approved_rcid_pairs = set(zip(rcid_decisions["gvkey_key"], rcid_decisions["rcid_key"]))
assert exported_rcid_pairs == approved_rcid_pairs
if approved_rcid_decisions.empty:
    assert crosswalk["rcid"].isna().all()
expected_history_pairs = sorted((int(row.membership_row), int(row.ccm_row)) for row in primary_links.itertuples(index=False))
actual_history_pairs = sorted((record["membership_row"], record["ccm_row"]) for value in crosswalk["historical_links_json"] for record in json.loads(value))
assert actual_history_pairs == expected_history_pairs
assert sum(len(json.loads(value)) for value in crosswalk["revelio_candidates_json"]) == len(candidate_method_sets)
display(pd.Series({
    "provisional_firm_rows": len(crosswalk), "membership_bridge_rows": len(membership_bridge),
    "preserved_pc_link_records": len(actual_history_pairs),
    "preserved_unresolved_intervals": len(expected_unresolved_ids),
    "explicitly_accepted_rcid": len(exported_rcid_pairs),
}, name="count").to_frame())
expected_bridge_gvkey = primary_links.groupby("membership_row")["gvkey_key"].agg(
    lambda values: tuple(sorted(values.dropna().unique()))
)
for row in membership_bridge.itertuples(index=False):
    keys = expected_bridge_gvkey.get(row.membership_row, ())
    if len(keys) == 1:
        assert row.candidate_gvkey == keys[0]
        assert row.candidate_firm_id == "gvkey:" + keys[0]
    else:
        assert pd.isna(row.candidate_gvkey) and pd.isna(row.candidate_firm_id)
assert membership_bridge["assignment_status"].isin([
    "pc_candidate_unverified", "ambiguous_pc_candidates", "no_pc_assignment_unresolved",
]).all()


### 7.5 Schema preparation

Define both outputs: the one-row-per-GVKEY provisional crosswalk and the complete
one-row-per-membership-interval bridge. Document source fields, nullable candidate
assignments, status flags, and nested evidence. The schema file is written only after
both staged Parquet files pass read-back. No identity acceptance is implied.

In [ ]:
field_definitions = {
    "firm_id": "Proposed key: gvkey: plus source GVKEY; not a legal-entity adjudication.",
    "gvkey": "CCM P/C candidate GVKEY; one row per key in the provisional universe.",
    "cik": "Unique normalized observed Compustat CIK; null if absent or multiple; not SEC-verified.",
    "cusip8": "Unique first eight characters of valid observed Compustat CUSIP; null if absent or multiple.",
    "permno": "Single P/C-linked PERMNO, otherwise null; all securities retained in history.",
    "ticker": "Single observed Compustat ticker, otherwise null; historical validity not established.",
    "ticker_valid_from": "Null: verified historical ticker start date unavailable.",
    "ticker_valid_to": "Null: verified historical ticker end date unavailable.",
    "rcid": "Only an explicitly researcher-accepted local RCID candidate; otherwise null.",
    "ultimate_parent_rcid": "Single Revelio parent snapshot for an accepted RCID; not historically validated.",
    "transcript_symbol": "Null: verified transcript identifier mapping unavailable.",
    "match_method": "Evidence methods for an accepted RCID; does not claim CIK-first precedence.",
    "match_score": "Null: no approved scoring model exists.",
    "sp500_entry": "Source start for a single membership interval without the early-date flag; otherwise null.",
    "sp500_exit": "Source end for a single membership interval, except sample-end dates; otherwise null.",
    "artifact_status": "Always provisional_candidate_crosswalk; not a fully resolved final crosswalk.",
    "rcid_status": "Unreviewed candidate, no local candidate, or accepted identity with time validity unverified.",
    "parent_validity_status": "Parent historical validity remains unverified.",
    "decision_basis": "Researcher-entered RCID acceptance reason; null without acceptance.",
    "method_category": "Part 6 mutually exclusive local candidate-method category.",
    "review_status": "Part 6 pre-adjudication diagnostic status, retained after explicit decisions.",
    "membership_interval_count": "Distinct source membership intervals in preserved P/C history.",
    "linked_permno_count": "Distinct P/C-linked securities for the GVKEY.",
    "historical_links_json": "Every P/C membership-link record, including source dates, E marker, and effective overlap.",
    "revelio_candidates_json": "All local RCID candidates and evidence methods; not accepted matches.",
}
field_definitions["ls_review"] = "Retained P/C LS evidence; flagged for review, not verified acceptance."
field_definitions["cross_company_mapping_review"] = "PERMCO/GVKEY multiplicity needing dated review; not proof of conflict."
for flag in firm_exception_flags:
    field_definitions[flag] = "Part 6 diagnostic flag: " + flag.replace("_", " ") + "; retained without adjudication."
assert set(field_definitions) == set(crosswalk.columns)
crosswalk_schema = pd.DataFrame([{
    "column": column, "pandas_dtype": str(crosswalk[column].dtype),
    "source_and_construction": field_definitions[column],
} for column in crosswalk.columns])
display(crosswalk_schema)

bridge_definitions = {
    "membership_row": "Unique original data-row number in the membership input; one interval per row.",
    "indno": "Original CRSP index identifier, string.",
    "permno": "Original CRSP security identifier, string.",
    "permco": "Original CRSP company identifier, string.",
    "mbrstartdt": "Unchanged original membership start string, including early dates.",
    "mbrenddt": "Unchanged original membership end string; sample-end is not confirmed departure.",
    "effective_start": "Membership start bounded by study start, from Part 2.",
    "effective_end": "Membership end bounded by study end, from Part 2.",
    "candidate_gvkey": "Single distinct date-overlapping P/C GVKEY; otherwise null, never sourced from J-only evidence.",
    "candidate_firm_id": "Provisional gvkey: key for a unique P/C candidate, not an accepted identity.",
    "linkprim": "Source marker for exactly one P/C record; otherwise null with JSON evidence.",
    "linktype": "Source type for exactly one P/C record; LS remains flagged.",
    "linkdt": "Original CCM start string for one P/C record, otherwise null.",
    "linkenddt": "Original CCM end string including E for one P/C record, otherwise null.",
    "overlap_start": "Effective overlap start for one P/C record, otherwise null.",
    "overlap_end": "Effective overlap end for one P/C record, otherwise null.",
    "pc_record_count": "Number of date-overlapping P/C candidate records with GVKEY.",
    "pc_gvkey_count": "Distinct P/C candidate GVKEY count.",
    "any_date_overlap": "Any date-overlapping CCM candidate with GVKEY, including secondary links.",
    "has_pc_overlap": "At least one P/C date-overlapping candidate with GVKEY; not acceptance.",
    "single_link_full_coverage": "A single P/C link spans the whole effective interval.",
    "pc_union_full_coverage": "Union of P/C overlap segments covers every inclusive calendar day of the interval.",
    "partial_coverage": "P/C overlap exists but its union does not cover the whole interval.",
    "j_only_evidence": "No P/C overlap; every date-overlapping candidate with GVKEY is J.",
    "one_day_boundary": "A candidate has one-day endpoint separation; original dates unchanged.",
    "ls_review": "A retained P/C candidate has LS type; requires review.",
    "ambiguous_pc_gvkey": "Multiple P/C candidate GVKEY; scalar assignment remains null.",
    "assignment_status": "Candidate-only P/C status, ambiguous candidates, or unresolved; never verified acceptance.",
    "unresolved_reason": "Coverage, identity ambiguity, boundary, or LS review reasons.",
    "ccm_candidates_json": "Every identifier-level source CCM candidate, including J/N and non-overlapping dates.",
}
assert set(bridge_definitions) == set(membership_bridge.columns)
bridge_schema = pd.DataFrame([{
    "column": column, "pandas_dtype": str(membership_bridge[column].dtype),
    "source_and_construction": bridge_definitions[column],
} for column in membership_bridge.columns])
display(bridge_schema)


### Findings

These findings refer to the saved 7.1-7.5 execution before the new bridge and quality checks.

- The provisional firm table contains 652 rows and unique `gvkey:` IDs. No RCID has been researcher-accepted; all accepted-RCID fields remain null.
- Five firms have multiple linked securities and seven have multiple membership intervals. All 659 P/C history records are retained without selecting one security or flattening intervals.
- Seven intervals without P/C overlap require separate preservation. The complete bridge and new LS/cross-firm diagnostics are prepared but their results remain pending researcher execution.
- Historical ticker-validity dates, transcript identifiers, and match scores remain unavailable. Parent identifiers represent snapshots, not historical assertions.
- This is not yet an accepted Revelio identity crosswalk. No Parquet export or read-back was executed during preparation.

### 7.6 staged export and read-back

Stage both Parquet files, schema, and revised log under data/clean on the same filesystem.
Read both staged tables with pyarrow, checking equality, dtypes, keys, source intervals,
unresolved rows, and P/C record identities. Publish only after validation, without
replacing existing output artifacts. Atomically replace the log last.

On a caught failure, remove only files published by this attempt. Preserve staging for
manual recovery if rollback fails. Multi-file publication is not globally atomic: a
hard process interruption can require recovery review. Repeated execution refuses
existing outputs; never remove or replace them without review. No final export success
is claimed until the researcher saves a successful read-back result.

In [ ]:
import os
import tempfile
import pyarrow as pa
import pyarrow.parquet as pq

if export_authorized is not True or not all(value is True for value in export_approvals.values()):
    raise RuntimeError("Provisional design and manual export approvals are required.")
assert len(crosswalk) == candidate_companies["gvkey_key"].nunique()
assert crosswalk["firm_id"].is_unique and crosswalk["gvkey"].is_unique
assert len(membership_bridge) == len(membership)
assert membership_bridge["membership_row"].is_unique
assert set(membership_bridge.loc[~membership_bridge["has_pc_overlap"], "membership_row"]) == expected_unresolved_ids
assert set(zip(crosswalk.loc[crosswalk["rcid"].notna(), "gvkey"], crosswalk.loc[crosswalk["rcid"].notna(), "rcid"])) == approved_rcid_pairs

output_directory = (project_root / "data/clean").resolve()
final_paths = [
    output_directory / "crosswalk.parquet",
    output_directory / "crosswalk_membership_bridge.parquet",
    output_directory / "crosswalk_schema.md",
]
research_log_path = output_directory / "crosswalk_log.md"
if any(output_path.exists() for output_path in final_paths):
    raise FileExistsError("Final output already exists. Review before requesting replacement; nothing overwritten.")
original_log_bytes = research_log_path.read_bytes()

schema_lines = [
    "# Crosswalk v1: Provisional Candidate Schema", "",
    "Main table: one row per P/C-candidate GVKEY; provisional firm_id = gvkey: + source GVKEY.",
    "Bridge: one row per original membership interval, including unresolved J/boundary evidence.",
    "Candidate assignments are not acceptance. Unapproved RCID remains null; parent values are snapshots.",
    "Annual-format selection, identity conflicts, and historical ticker validity remain unresolved.", "",
]
for title, schema in [("crosswalk.parquet", crosswalk_schema), ("crosswalk_membership_bridge.parquet", bridge_schema)]:
    schema_lines.extend(["## " + title, "", "| Column | pandas dtype | Source and construction |", "|---|---|---|"])
    for row in schema.itertuples(index=False):
        schema_lines.append(f"| {row.column} | {row.pandas_dtype} | {row.source_and_construction} |")
    schema_lines.append("")

stage_directory = Path(tempfile.mkdtemp(prefix=".crosswalk_stage_", dir=output_directory))
assert stage_directory.resolve().parent == output_directory
staged_paths = [stage_directory / output_path.name for output_path in final_paths]
staged_log_path = stage_directory / "crosswalk_log.md"
published_paths = []
export_succeeded = False
recovery_errors = []
try:
    pq.write_table(pa.Table.from_pandas(crosswalk, preserve_index=False), staged_paths[0])
    pq.write_table(pa.Table.from_pandas(membership_bridge, preserve_index=False), staged_paths[1])
    read_back_crosswalk = pq.read_table(staged_paths[0]).to_pandas()
    read_back_bridge = pq.read_table(staged_paths[1]).to_pandas()
    pd.testing.assert_frame_equal(crosswalk, read_back_crosswalk, check_dtype=True)
    pd.testing.assert_frame_equal(membership_bridge, read_back_bridge, check_dtype=True)
    assert len(read_back_crosswalk) == len(crosswalk) and read_back_crosswalk["firm_id"].is_unique
    assert len(read_back_bridge) == len(membership) and read_back_bridge["membership_row"].is_unique
    pd.testing.assert_frame_equal(
        read_back_bridge.set_index("membership_row")[original_interval_columns].sort_index(),
        membership.set_index("membership_row")[original_interval_columns].sort_index(), check_index_type=False,
    )
    assert set(read_back_bridge.loc[~read_back_bridge["has_pc_overlap"], "membership_row"]) == expected_unresolved_ids
    assert read_back_bridge.loc[~read_back_bridge["has_pc_overlap"], ["candidate_gvkey", "candidate_firm_id"]].isna().all(axis=None)
    restored_history_pairs = sorted((record["membership_row"], record["ccm_row"]) for value in read_back_crosswalk["historical_links_json"] for record in json.loads(value))
    assert restored_history_pairs == expected_history_pairs
    staged_paths[2].write_text(chr(10).join(schema_lines) + chr(10), encoding="utf-8")
    export_log_lines = [
        "", "## Provisional v1 export verified", "",
        f"UTC execution time: {pd.Timestamp.now(tz='UTC').isoformat()}; pyarrow {pa.__version__}.",
        f"Validated {len(crosswalk)} firm rows and {len(membership_bridge)} original membership intervals.",
        f"Preserved {len(restored_history_pairs)} P/C history records and {len(expected_unresolved_ids)} unresolved non-P/C intervals.",
        f"Explicitly accepted RCID: {int(crosswalk['rcid'].notna().sum())}. No other candidates accepted.",
        "Both staged tables passed row/key checks and full equality including dtypes before publication.",
        "Design is provisional; unresolved historical and identity flags remain in the artifacts.", "",
    ]
    staged_log_path.write_bytes(original_log_bytes + chr(10).join(export_log_lines).encode("utf-8"))
    if research_log_path.read_bytes() != original_log_bytes:
        raise RuntimeError("Research log changed during preparation. Reconcile before exporting.")
    # Same-filesystem hard links fail rather than replace an existing final file.
    for staged_path, final_path in zip(staged_paths, final_paths):
        os.link(staged_path, final_path)
        published_paths.append((staged_path, final_path))
    if research_log_path.read_bytes() != original_log_bytes:
        raise RuntimeError("Research log changed during publication; rolling back this attempt.")
    os.replace(staged_log_path, research_log_path)
    export_succeeded = True
except BaseException:
    for staged_path, final_path in reversed(published_paths):
        try:
            if final_path.exists():
                if not os.path.samefile(staged_path, final_path):
                    raise RuntimeError("Published file changed; refusing to remove it: " + str(final_path))
                final_path.unlink()
        except (OSError, RuntimeError) as recovery_error:
            recovery_errors.append(str(recovery_error))
    if recovery_errors:
        print("Manual recovery required; staging retained at:", stage_directory)
        print(recovery_errors)
    raise
finally:
    if not recovery_errors:
        try:
            # Clean only this attempt's known staging files; no recursive deletion.
            assert stage_directory.resolve().parent == output_directory
            for staged_path in staged_paths + [staged_log_path]:
                if staged_path.exists():
                    staged_path.unlink()
            stage_directory.rmdir()
        except OSError as cleanup_error:
            print("Staging cleanup requires inspection:", stage_directory, cleanup_error)

if export_succeeded:
    print("Both provisional Parquet files passed read-back; schema and log published.")
    print("Firm rows:", len(read_back_crosswalk), "| Membership intervals:", len(read_back_bridge))
    print("Unresolved intervals retained:", len(expected_unresolved_ids))
    print("Outputs:", [output_path.name for output_path in final_paths])